<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L04-adverse-action-reason-codes/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L04-adverse-action-reason-codes/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/credit-underwriting/lessons/P05-L04-adverse-action-reason-codes/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/credit-underwriting/lessons/P05-L04-adverse-action-reason-codes/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P05-L04 · Adverse action and reason codes: a stable, non-proxy reason for every decline

**You will build:** the step that turns a declined application into the principal reasons a
notice must state, two ways: from the **points of a scorecard**, the way the regulation's own
commentary describes, and from **exact Shapley attributions** of a boosted challenger. Then you
write the tests that decide whether those reasons are fit to send: does the reason change when
an input is recorded a hair differently, does it change when only the reference population
changes, and does it behave like a stand-in for a protected group?

**Time:** ~3 hours · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (what a declared compute budget means),
`P04-L08-explainability-evidence` (exact Shapley attributions by enumeration, carried into this
notebook as given code) and `P05-L03-scorecard-versus-boosting` (the weight-of-evidence
scorecard and the monotone-constrained booster, carried in as given code). Python and numpy; no
pandas, no scikit-learn and no SHAP library: everything is in this notebook.

By the end you will be able to:

1. Implement integer scorecard points, the reference population of Regulation B's commentary
   (applicants at or just above the cut-off, or all applicants) and the choice of principal
   reasons with a stated tie rule.
2. Measure that the points-below-reference method and exact Shapley attributions give the same
   numbers for an additive scorecard, and use Shapley attributions for a boosted challenger that
   is not additive.
3. Implement a perturbation test and measure how often a declined applicant's reasons change
   when an input moves by a negligible amount or when only the reference population changes,
   read against the error a sampled background makes.
4. Implement reason codes that sum the shortfalls of inputs sharing a cause, and measure what a
   twin input does to a list of reasons that is built input by input.
5. Implement citation counts by group and a proxy-like screen, and measure which reasons each
   model gets flagged, what dropping a flagged input costs, and how the audit that combines the
   pieces reads for four configurations.

**This is engineering, not legal advice.** The lesson quotes the regulation as it read on
19 October 2026 and says what each test measures. It never says that a reason, an input or a
model is lawful or unlawful: that turns on facts and on law this notebook cannot see, and the
text of the regulation changed earlier this year.

**The data are synthetic and the generator is in this notebook.** Nothing is downloaded and no
real applicant, lender or decision is represented. A planted truth is what lets the notebook say
which relationships are in the world and which are not, which no real book can tell you. What
you conclude here about *method* transfers; what you conclude about *these numbers* describes
no real lender.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `P04-L08`, `P05-L03`

**This lesson assumes you can already:**

- write numpy code that sorts along an axis, indexes with boolean masks, and sums or counts along an axis
- explain what a scorecard's weight of evidence is, and read a Gini coefficient as 2 AUC minus 1 (P04-L07-challenger-from-scratch and P04-L04-discrimination-testing teach them; this lesson introduces its points scale itself)
- say what a Shapley value is: one input's average marginal contribution over every order in which the inputs could be revealed, and why the values add up to the prediction minus the baseline
- read a boosted tree model as a sum of small decision trees whose leaves hold weights (P05-L03-scorecard-versus-boosting builds the booster that this lesson carries in as given code)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [12 CFR 1002.9, Notifications (Regulation B)](https://www.ecfr.gov/current/title-12/chapter-X/part-1002/subpart-A/section-1002.9) · Consumer Financial Protection Bureau; the text as published in the Electronic Code of Federal Regulations · *free* — paragraph (b)(2) is the sentence the whole lesson tests its reasons against: specific, principal, and not merely 'failed to achieve a qualifying score'
- [Supplement I to Part 1002, Official Interpretations (Regulation B)](https://www.ecfr.gov/current/title-12/chapter-X/part-1002/appendix-Supplement%20I%20to%20Part%201002) · Consumer Financial Protection Bureau; the text as published in the Electronic Code of Federal Regulations · *free* — the comments on 1002.9(b)(2) set out the two ways of choosing reasons from a scoring system that this lesson builds, and the comments on 1002.6(a) are the passage the proxy screen is read against
- [Interpretable Machine Learning, chapter 'Shapley Values'](https://christophm.github.io/interpretable-ml-book/shapley.html) · Christoph Molnar (free online book) · *free* — the definition of the Shapley value and its efficiency property, which this lesson checks numerically, and the discussion of the background data that plays the part of the reference population
- [A Unified Approach to Interpreting Model Predictions](https://arxiv.org/abs/1705.07874) · Lundberg and Lee (NeurIPS 2017, arXiv) · *free* — the additive feature attribution view and the local accuracy property (attributions sum to the output minus a baseline) behind the audit's check that every decline has a reason
- [numpy.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) · The NumPy developers (documentation) · *free* — the stable sort that exercise 3's tie rule depends on: equal shortfalls must keep their input order

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import sys
import time
import traceback
from math import factorial
from typing import Any, Callable, NamedTuple

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

# The book. SEED is the date this lesson was written.
SEED = 20261019
N_DEV, N_VAL, N_DEC = 14_000, 4_000, 8_000   # development, validation and decision rows
FEATURES = ("utilisation", "age of oldest account", "delinquencies", "enquiries", "area index")
SHORT = ("utilisation", "age", "delinq.", "enquiries", "area")      # column headings
P = len(FEATURES)
AREA = 4                 # the column of the area index
CONTINUOUS = (0, 1, 4)   # inputs recorded on a continuous scale; the two counts are recorded exactly
# The documented direction of RISK for each input: +1 means predicted risk must never fall as the
# input rises, -1 that it must never rise. Written down before any model was fitted.
DIRECTIONS = np.array([1, -1, 1, 1, 1])

# The points scale, and the policy.
PDO = 20                 # points that double the good:bad odds
BASE_POINTS, BASE_ODDS = 600, 50      # 600 points at 50 good for every bad
FACTOR = PDO / math.log(2.0)
OFFSET = BASE_POINTS - FACTOR * math.log(BASE_ODDS)
DECLINE_SHARE = 0.25     # the cut-off declines this share of the development applicants
MARGINAL_WIDTH = 10      # "slightly above the minimum passing score": this many points above the cut
NOTICE_REASONS = 2       # reasons stated on a notice
MAX_REASONS = 4          # the most the lesson ever asks for (the regulation's commentary: more is not helpful)
NO_REASON = -1           # padding when an applicant has fewer reasons than the notice has places

# The scorecard.
WOE_BINS = 8             # quantile classes for the continuous inputs, before pooling
PSI_FLOOR = 1e-6
COUNT_CLASSES = {2: np.array([-np.inf, 1.0, 2.0, 3.0, 4.0, np.inf]),
                 3: np.array([-np.inf, 1.0, 2.0, 3.0, 4.0, 6.0, np.inf])}

# The boosted trees (the settings of P05-L03, except the first cap on the number of trees, which is 100 here).
N_BINS = 32
LEARNING_RATE = 0.1
MAX_DEPTH = 3
LAMBDA = 1.0
MIN_CHILD_HESS = 3.0
MAX_ROUNDS = 100          # a first cap on the trees; the validation rows pick fewer, and a fit that hits the cap is refitted with more
HARD_ROUNDS = 600
HESS_FLOOR = 1e-12

# Shapley attributions: the sizes of sampled backgrounds the lesson compares with the whole population.
N_BG = 200
BG_SWEEP = (25, 50, 100, 200, 400)
N_SAMPLES = 12            # equally valid samples of each size: the spacing starts at a different row each time

REPLICATES = 4                        # books per sweep: one book is one draw
# The tests. These numbers are this lesson's own choices, not regulatory numbers.
EPS_MAIN = 0.01                       # a "negligible" move, in development standard deviations
EPS_SWEEP = (0.001, 0.003, 0.01, 0.03, 0.1)
STABILITY_TOL = 0.05                  # the reason set may change for at most this share of declines
MIN_RATIO = 1.5                       # citation rate of group 1 over group 0
MIN_GROUP_GINI = 0.25                 # how well the input alone separates the two groups
MIN_CITED = 30                        # fewer citations than this say nothing

# What a notice calls each reason (comment 9(b)(2)-3: it need not say how a factor hurt).
REASON_TEXT = ("Utilisation of revolving credit", "Age of oldest account",
               "Number of delinquencies", "Number of recent enquiries", "Credit index of the applicant's area")

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("class_points",),
    "exercise 2": ("reference_mask",),
    "exercise 3": ("principal_reasons",),
    "exercise 4": ("reasons_differ",),
    "exercise 5": ("code_shortfall",),
    "exercise 6": ("perturbation_flip_rate",),
    "exercise 7": ("citation_counts",),
    "exercise 8": ("proxy_flags",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
S: dict[str, Any] = {}         # results the demos hand to one another
_PRODUCER = {"card": "the scorecard fit", "decision": "the scorecard's points",
             "ref_means": "the reference populations", "ref_exact": "the reference populations", "card_reasons": "the scorecard's reasons",
             "challenger": "the challenger", "boost": "the challenger's reasons", "sampling": "the sampled backgrounds", "books": "the replicate books",
             "group_gini": "the scorecard's proxy screen", "refit": "the refitted models"}

# Heavy demos (the booster, exact attributions for hundreds of applicants, refits) run when this
# file is the program, in a script or in a notebook kernel, and are skipped when a grader merely
# imports it to test your functions.
_RUNNING = __name__ == "__main__"


def _book_seed(k: int) -> int:
    """The seed of replicate book k (k = 0 is the book the lesson walks through)."""
    return SEED + 7919 * k


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (principal_reasons)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _ready(demo: str, *keys: str) -> bool:
    """True when the cells that fill `S[key]` have finished; otherwise name them and say the demo is skipped."""
    missing = sorted({_PRODUCER[k] for k in keys if k not in S})
    if not missing:
        return True
    print(f"{demo}: skipped — it needs the output of {', '.join(missing)}, which did not finish.")
    return False


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 180     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 1024     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (), heavy: bool = False) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero. A `heavy` demo is skipped when this file is imported rather than run.
    """
    if heavy and not _RUNNING:
        return
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def sigmoid(x: Any) -> Any:
    """Numerically safe logistic function 1 / (1 + exp(-x)). Given to you."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out[()]


def auc_gini(score: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Tie-aware AUC and Gini (2 * AUC - 1), higher score = more likely y = 1. Given to you.

    The rank form of the AUC from P04-L04 (carried into P04-L07): ranks with ties sharing the
    average rank, so a tied pair counts one half. Raises ValueError for one class, unequal lengths
    or a y that is not 0/1.
    """
    s = np.asarray(score, dtype=float)
    t = np.asarray(y)
    if s.ndim != 1 or t.ndim != 1 or s.size != t.size:
        raise ValueError("score and y must be 1-D arrays of equal length")
    if not np.all((t == 0) | (t == 1)):
        raise ValueError("y must contain only 0 and 1")
    n1, n0 = int(np.sum(t == 1)), int(np.sum(t == 0))
    if n1 == 0 or n0 == 0:
        raise ValueError("y must contain both classes")
    order = np.argsort(s, kind="stable")
    ordered = s[order]
    starts = np.flatnonzero(np.concatenate(([True], ordered[1:] != ordered[:-1])))
    ends = np.concatenate((starts[1:], [s.size]))
    ranks = np.empty(s.size)
    ranks[order] = np.repeat((starts + ends + 1) / 2.0, ends - starts)
    auc = (ranks[t == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
    return float(auc), float(2.0 * auc - 1.0)


def systematic_sample(rows: np.ndarray, m: int, offset: int = 0) -> np.ndarray:
    """`m` of the entries of `rows`, evenly spaced, starting `offset` entries in. Given.

    Rows of the book are in random order, so every k-th row is a fair sample; it is also the same
    sample on every machine, which a random draw would not be unless it were seeded.
    """
    rows = np.asarray(rows)
    step = max(len(rows) // m, 1)
    return rows[offset: offset + m * step: step]

## 1. What a notice must say, and a book in which the truth is known

These are the passages of 12 CFR part 1002 (Regulation B) this lesson builds on, as the eCFR
showed them when the lesson was written: the text of 15 October 2026, the latest the service had on 19 October 2026. `claims.yaml` has each one with where
it was read. **Re-read the current text before you rely on any of it:** the part was amended on
22 April 2026.

- **Section 1002.9(a)(2):** the notice of adverse action is in writing and carries "either: (i) A
  statement of specific reasons for the action taken; or (ii) A disclosure of the applicant's right
  to a statement of specific reasons within 30 days".
- **Section 1002.9(b)(2):** the statement of reasons "must be specific and indicate the
  principal reason(s) for the adverse action", and saying only that the applicant "failed to
  achieve a qualifying score" is insufficient.
- **Comment 9(b)(2)-5:** no one method of choosing reasons is required. One is to take the
  factors where the applicant's score "fell furthest below the average score" of applicants "at
  or slightly above the minimum passing score"; another is the same against the average of all
  applicants; a method with "substantially similar" results is also acceptable.
- **Comment 9(b)(2)-1:** no set number of reasons, but more than four "is not likely to be
  helpful". **Comment 9(b)(2)-4:** the reasons relate only to factors actually scored, and "no
  factor that was a principal reason for adverse action may be excluded from disclosure".
- **Comment 9(b)(2)-9:** the Fair Credit Reporting Act's disclosure of the key factors behind a
  credit score is a different disclosure, and it "does not satisfy the ECOA requirement to
  disclose specific reasons". This lesson builds the ECOA reasons and nothing of the FCRA disclosure.

Two more passages matter for the last third of the notebook. Section 1002.6(a) now says "The
Act does not provide that the "effects test" applies", and comment 6(a)-2 says the Act does not
prohibit facially neutral practices "except to the extent that" the criteria "function as proxies
for protected characteristics designed or applied with the intention of" advantaging or
disadvantaging individuals. So the screen you write in exercise 8 measures a pattern and is
evidence for a human review; it says nothing about intent, and this lesson never reads it as a verdict.

The book below has five inputs and an outcome. It also has a **group label that no model ever
sees**. A fair-lending test needs one; where a lender does not hold it, module 5 of this
programme estimates it. Here it is simply planted, together with a **latent regional economy**
that really does move default risk and is not an input. The **area index** is built from that
region *and* from the group label, so it is a genuinely useful predictor that also carries the
group. The group itself never enters the default probability (it only shifts utilisation by a
small amount, a legitimate difference that a good screen should leave alone).

In [ ]:
class Book(NamedTuple):
    X: np.ndarray        # (n, 5) utilisation, age of oldest account (months), delinquencies, enquiries, area index
    y: np.ndarray        # (n,) 1 = defaulted within the outcome window, 0 = repaid
    group: np.ndarray    # (n,) 1 = member of the group under test: NOT an input to any model
    pd_true: np.ndarray  # (n,) the planted probability of default: for checking, never for fitting


B0 = -4.3                # intercept of the planted log-odds
GROUP_SHARE = 0.30       # share of applicants in the group under test
UTIL_SHIFT = 0.03        # the group's small, legitimate difference in utilisation
REGION_EFFECT = 0.5      # planted log-odds per standard deviation of the latent regional economy


def true_logit(X: np.ndarray, region: np.ndarray) -> np.ndarray:
    """The planted log-odds of default. A function of the four ordinary inputs and the region,
    never of the group and never of the area index itself."""
    u, age, d, q = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return (B0 + 2.4 * u + 1.6 * u ** 3 - 0.55 * np.log(age / 72.0)
            + 0.55 * np.sqrt(np.minimum(d, 6.0)) + 0.10 * np.minimum(q, 8.0)
            + 1.1 * np.sqrt(np.minimum(d, 4.0)) * np.exp(-age / 36.0) + REGION_EFFECT * region)


def application_book(seed: int) -> Book:
    """Deterministic book of N_DEV + N_VAL + N_DEC applications, in that order. Same seed, same book."""
    rng = np.random.default_rng(seed)
    n = N_DEV + N_VAL + N_DEC
    group = (rng.random(n) < GROUP_SHARE).astype(int)
    region = rng.standard_normal(n)                      # the latent regional economy
    stress = rng.standard_normal(n)                      # a latent factor the credit inputs share
    z_util = 0.5 * stress + np.sqrt(0.75) * rng.standard_normal(n)
    util = np.clip(0.40 + 0.20 * z_util + UTIL_SHIFT * group, 0.0, 1.2)
    age = 72.0 * np.exp(0.65 * rng.standard_normal(n))
    delinq = rng.poisson(0.22 * np.exp(0.5 * stress)).astype(float)
    enq = rng.poisson(1.2 * np.exp(0.3 * stress)).astype(float)
    area = 0.7 * region + 0.9 * (group - GROUP_SHARE) + 0.5 * rng.standard_normal(n)
    X = np.column_stack([util, age, delinq, enq, area])
    pd_true = sigmoid(true_logit(X, region))
    y = (rng.random(n) < pd_true).astype(int)
    return Book(X, y, group, pd_true)


BOOK = application_book(SEED)
DEV = slice(0, N_DEV)
VAL = slice(N_DEV, N_DEV + N_VAL)
DEC = slice(N_DEV + N_VAL, N_DEV + N_VAL + N_DEC)
X_DEV, Y_DEV, G_DEV = BOOK.X[DEV], BOOK.y[DEV], BOOK.group[DEV]
X_DEC, Y_DEC, G_DEC = BOOK.X[DEC], BOOK.y[DEC], BOOK.group[DEC]
SD_DEV = X_DEV.std(axis=0)         # each input's development standard deviation: the unit of a "negligible" move


def _show_book() -> None:
    print(f"{'period':<13}{'rows':>8}{'defaults':>10}{'rate':>8}{'group 1':>9}{'rate g0':>9}{'rate g1':>9}{'planted Gini':>14}")
    for name, sl in (("development", DEV), ("validation", VAL), ("decision", DEC)):
        y, g, p = BOOK.y[sl], BOOK.group[sl], BOOK.pd_true[sl]
        print(f"{name:<13}{len(y):>8,}{int(y.sum()):>10,}{y.mean():>8.1%}{g.mean():>9.1%}"
              f"{y[g == 0].mean():>9.1%}{y[g == 1].mean():>9.1%}{auc_gini(p, y)[1]:>14.3f}")
    p = BOOK.pd_true[DEC]
    print(f"\nplanted default probability on the decision rows, by group: group 0 {p[G_DEC == 0].mean():.2%}, "
          f"group 1 {p[G_DEC == 1].mean():.2%} (in this generator the group enters the default probability only through its utilisation shift)")
    print("'planted Gini' is the Gini of the planted probability on that period's outcomes: what a model")
    print("that knew the function exactly would score. In expectation no fitted model can beat it on new rows.")


_show_book()

## 2. The scorecard, carried in from P05-L03

The champion is the weight-of-evidence scorecard of `P04-L07` and `P05-L03`: every input cut into
classes, each class given its **weight of evidence** (the log of the non-defaulters' share over
the defaulters' share, so a higher weight is a safer class), neighbouring classes **pooled**
whenever their default rates move against the input's documented direction, and a logistic
regression on the weights. Its code is below, **given to you**: this lesson is about the reasons
the model gives, not about fitting it. One detail differs from `P05-L03` and it does not change
the method: the fit takes the directions and the analyst's count classes as arguments, so the
notebook can refit it on other columns later.

What matters for what follows: the score is a **sum of one term per input** (a class lookup
times a coefficient), so each input's contribution to the score is a number you can read off.
Exercise 1 turns those terms into whole points; the rest of the lesson asks what to do with them.

In [ ]:
def quantile_edges(x: np.ndarray, n_bins: int) -> np.ndarray:
    """Quantile edges of `x`, de-duplicated, with the outer edges opened to infinity. Given."""
    edges = np.unique(np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


def woe_from_counts(events: np.ndarray, non_events: np.ndarray, smoothing: float = 0.5) -> np.ndarray:
    """Weight of evidence per class: ln of the non-events' smoothed share over the events'. Given.

    With a = smoothing and m the number of occupied classes, an occupied class's weight is
    ln(((non_i + a) / (N_non + a m)) / ((ev_i + a) / (N_ev + a m))); an empty class is nan.
    """
    e, g = np.asarray(events, dtype=float), np.asarray(non_events, dtype=float)
    occ = (e + g) > 0
    m = int(occ.sum())
    woe = np.full(e.size, np.nan)
    woe[occ] = np.log(((g[occ] + smoothing) / (g.sum() + smoothing * m))
                      / ((e[occ] + smoothing) / (e.sum() + smoothing * m)))
    return woe


class WoeTable(NamedTuple):
    """One input's classes. The last entry of each array is the MISSING class (empty here)."""
    edges: np.ndarray       # value-class edges
    events: np.ndarray      # defaults per value class, then the missing class
    non_events: np.ndarray
    woe: np.ndarray


def class_index(table: WoeTable, x: np.ndarray) -> np.ndarray:
    """The value class of each entry of `x` (a value on an interior edge joins the upper class). Given."""
    n_value = table.edges.size - 1
    return np.clip(np.searchsorted(table.edges, np.asarray(x, dtype=float), side="right") - 1, 0, n_value - 1)


def woe_table(x: np.ndarray, y: np.ndarray, edges: np.ndarray) -> WoeTable:
    """Class `x` on `edges` and weigh each class. Given."""
    xa, ya = np.asarray(x, dtype=float), np.asarray(y)
    n_value = edges.size - 1
    miss = np.isnan(xa)
    idx = np.clip(np.searchsorted(edges, xa[~miss], side="right") - 1, 0, n_value - 1)
    yv = ya[~miss]
    events = np.append(np.bincount(idx, weights=(yv == 1).astype(float), minlength=n_value),
                       (ya[miss] == 1).sum()).astype(np.int64)
    non_events = np.append(np.bincount(idx, weights=(yv == 0).astype(float), minlength=n_value),
                           (ya[miss] == 0).sum()).astype(np.int64)
    return WoeTable(edges, events, non_events, woe_from_counts(events, non_events))


def enforce_monotonic(table: WoeTable, direction: str) -> WoeTable:
    """Pool adjacent value classes until the default rate moves only in `direction`. Given.

    "increasing" means the default rate may never fall from one class to the next; a merged class is
    compared with the one before it again (pooling adjacent violators); the missing class is untouched.
    """
    ev, ne, edges = table.events, table.non_events, table.edges
    n_value = edges.size - 1
    sign = 1.0 if direction == "increasing" else -1.0
    blocks: list[list[int]] = []          # [events, non-events, index of the closing edge]
    for i in range(n_value):
        blocks.append([int(ev[i]), int(ne[i]), i + 1])
        while len(blocks) >= 2:
            (e1, n1, _), (e2, n2, end) = blocks[-2], blocks[-1]
            if sign * (e2 / (e2 + n2) - e1 / (e1 + n1)) < 0.0:
                blocks[-2:] = [[e1 + e2, n1 + n2, end]]
            else:
                break
    new_edges = np.concatenate(([edges[0]], [edges[b[2]] for b in blocks]))
    events = np.array([b[0] for b in blocks] + [int(ev[-1])], dtype=np.int64)
    non_events = np.array([b[1] for b in blocks] + [int(ne[-1])], dtype=np.int64)
    return WoeTable(new_edges, events, non_events, woe_from_counts(events, non_events))


def apply_woe(x: np.ndarray, table: WoeTable) -> np.ndarray:
    """Replace each value by its class's weight of evidence. Given."""
    return table.woe[:table.edges.size - 1][class_index(table, x)]


def fit_logistic(X: np.ndarray, y: np.ndarray, ridge: float = 1e-6) -> np.ndarray:
    """Logistic regression by Newton's method with a tiny ridge; intercept first. Given."""
    A = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    yv = np.asarray(y, dtype=float)
    coef = np.zeros(A.shape[1])
    for _ in range(50):
        p = sigmoid(A @ coef)
        grad = A.T @ (p - yv) + ridge * coef
        hess = (A * (p * (1.0 - p))[:, None]).T @ A + ridge * np.eye(A.shape[1])
        step = np.linalg.solve(hess, grad)
        coef = coef - step
        if np.abs(step).max() < 1e-9:
            break
    return coef


class Scorecard(NamedTuple):
    tables: tuple          # one pooled WoeTable per input
    coef: np.ndarray       # intercept first, then one coefficient per input (on its weights of evidence)


def fit_scorecard(X: np.ndarray, y: np.ndarray, directions: np.ndarray,
                  count_classes: dict = COUNT_CLASSES) -> Scorecard:
    """Fit the champion: classes, weights of evidence, pooling to the documented direction, logistic. Given."""
    tables = []
    for j in range(X.shape[1]):
        edges = count_classes[j] if j in count_classes else quantile_edges(X[:, j], WOE_BINS)
        raw = woe_table(X[:, j], y, edges)
        tables.append(enforce_monotonic(raw, "increasing" if directions[j] > 0 else "decreasing"))
    design = np.column_stack([apply_woe(X[:, j], tables[j]) for j in range(X.shape[1])])
    return Scorecard(tuple(tables), fit_logistic(design, y))


def _fit_card() -> None:
    card = fit_scorecard(X_DEV, Y_DEV, DIRECTIONS)
    S["card"] = card
    print(f"{'input':<24}{'direction':>11}{'classes':>9}{'coefficient':>13}")
    for j, name in enumerate(FEATURES):
        print(f"{name:<24}{('rises' if DIRECTIONS[j] > 0 else 'falls'):>11}"
              f"{card.tables[j].edges.size - 1:>9}{card.coef[1 + j]:>13.3f}")
    wrong = [FEATURES[j] for j in range(P) if card.coef[1 + j] > 0.0]
    print("\na higher weight of evidence means a safer class, so every coefficient should be negative;")
    print(f"coefficients with the wrong sign on this book: {', '.join(wrong) if wrong else 'none'}.")


_fit_card()

## 3. Exercise 1 — `class_points()`

The scorecard's log-odds of *repaying* is a sum, with one term per input: the input's coefficient
(negative here, because a higher weight of evidence is a safer class and the model predicts
*default*) times the weight of its class, with its sign turned round. Put that term on the
points scale of this notebook (`FACTOR` points per unit of log-odds, so that `PDO` points double
the odds) and round it to a **whole number of points**: a scorecard document, a notice and the
compiled decision engine planned for this programme all work in whole points.

Whole points make **ties** possible, and a tie decides which reason an applicant is given (a later
cell counts them), so the rounding rule has to be exact. This lesson's rule: the points of a class are
`floor(factor * (-coef) * woe + 0.5)`, computed in that order, so a value exactly halfway goes
**up** (2.5 becomes 3, and -2.5 becomes -2).

<details><summary>💡 Hint 1 — what to think about</summary>

Three things can go wrong, and they are three different mistakes. Python's `round` and numpy's
`np.round` send a halfway value to the nearest *even* number. Converting to an integer with
`int` or `astype` truncates towards zero, which moves negative values the wrong way. And
"halfway rounds away from zero" is a third rule that differs from this one on negative halves.
The check below has a halfway value of each sign.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate first: the weights must be a 1-D array of finite numbers (an empty class has a weight
of nan and no points) and the factor must be a positive finite number. Then form the products
in the order the text gives, add one half, take the floor, and return whole numbers as a numpy
integer array.

</details>

In [ ]:
def class_points(coef: float, woe: np.ndarray, factor: float) -> np.ndarray:
    """Whole points for each class of one input: floor(factor * (-coef) * woe + 0.5).

    Requirements, each graded:
      * the products are formed left to right as written, then 0.5 is added and the floor taken:
        a value exactly halfway between two whole numbers goes UP (0.5 -> 1, -0.5 -> 0, -2.5 -> -2);
      * the result is a numpy integer array, one entry per class, in the order given;
      * `ValueError` if `woe` is not a 1-D array of finite numbers (a nan is an empty class), if
        `coef` is not finite, or if `factor` is not a positive finite number.

    Example (a point scale of 5 points per unit of weight; the halfway values 2.5 and -2.5):
        >>> class_points(-0.5, np.array([1.0, 0.5, -0.5, -1.5]), 10.0).tolist()
        [5, 3, -2, -7]

    Returns:
        a numpy int64 array.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_class_points() -> None:
    got = np.asarray(class_points(-0.5, np.array([1.0, 0.5, -0.5, -1.5]), 10.0))
    assert got.tolist() == [5, 3, -2, -7], (
        f"got {got.tolist()} for the docstring's example, expected [5, 3, -2, -7]: the points are "
        "floor(factor * (-coef) * woe + 0.5), so 2.5 becomes 3 and -2.5 becomes -2")
    halves = np.asarray(class_points(-1.0, np.array([0.5, 1.5, 2.5, -0.5, -1.5, -2.5]), 1.0))
    assert halves.tolist() == [1, 2, 3, 0, -1, -2], (
        f"values exactly halfway gave {halves.tolist()}; with floor(v + 0.5) a half always goes up: "
        "[1, 2, 3, 0, -1, -2]. np.round sends halves to the even number (0.5 -> 0, 2.5 -> 2); "
        "int() truncates towards zero; rounding away from zero sends -0.5 to -1")
    assert np.issubdtype(halves.dtype, np.integer), "return whole numbers as an integer array, not floats"
    plain = np.asarray(class_points(-0.8, np.array([0.3, -0.3, 1.1, -2.2]), 4.0))
    want = [math.floor(4.0 * 0.8 * v + 0.5) for v in (0.3, -0.3, 1.1, -2.2)]
    assert plain.tolist() == want, f"got {plain.tolist()}, expected {want} for non-halfway values"
    for bad in ((-0.5, np.array([1.0, np.nan]), 10.0), (-0.5, np.array([[1.0]]), 10.0),
                (-0.5, np.array([1.0]), 0.0), (float("nan"), np.array([1.0]), 10.0)):
        try:
            class_points(*bad)
        except ValueError:
            continue
        raise AssertionError("a nan weight (an empty class), a 2-D array, a factor that is not positive or a nan "
                             "coefficient must raise ValueError")
    print("exercise 1 looks right")

In [ ]:
_try("exercise 1", _check_class_points)

The next cell is given. It turns the scorecard into whole points for any rows, adds the
intercept's points, cuts the development totals at the policy's decline share, and applies
that cut to the decision rows. Run the cell after it to see the points.

In [ ]:
class Decision(NamedTuple):
    points_dev: np.ndarray    # (n, p) whole points per input, development rows
    total_dev: np.ndarray     # (n,) intercept points plus the sum of the inputs' points
    points_dec: np.ndarray    # the same for the decision rows
    total_dec: np.ndarray
    cut: float                # a total below the cut is declined
    declined: np.ndarray      # (n_dec,) boolean


def class_table_points(card: Scorecard, j: int) -> np.ndarray:
    """The whole points of each value class of input `j`. Given; uses your class_points."""
    table = card.tables[j]
    return class_points(float(card.coef[1 + j]), table.woe[:table.edges.size - 1], FACTOR)


def scorecard_points(card: Scorecard, X: np.ndarray) -> np.ndarray:
    """(n, p) whole points of each input for every row of raw inputs. Given; uses your class_points."""
    X = np.asarray(X, dtype=float)
    return np.column_stack([class_table_points(card, j)[class_index(card.tables[j], X[:, j])]
                            for j in range(len(card.tables))])


def intercept_points(card: Scorecard) -> int:
    """The points that go with the intercept, rounded by the same rule as every class. Given."""
    return int(math.floor(OFFSET - FACTOR * float(card.coef[0]) + 0.5))


def cut_from_development(total: np.ndarray, share: float) -> float:
    """The total of the applicant ranked `share` of the way up the development totals; lower is declined. Given."""
    ordered = np.sort(np.asarray(total))
    return ordered[int(share * len(ordered))]


def decide(card: Scorecard, X_dev: np.ndarray, X_dec: np.ndarray) -> Decision:
    """Points, totals, the cut-off cut on development and the declined decision rows. Given."""
    base = intercept_points(card)
    p_dev, p_dec = scorecard_points(card, X_dev), scorecard_points(card, X_dec)
    t_dev, t_dec = base + p_dev.sum(axis=1), base + p_dec.sum(axis=1)
    cut = cut_from_development(t_dev, DECLINE_SHARE)
    return Decision(p_dev, t_dev, p_dec, t_dec, cut, t_dec < cut)


def _show_points() -> None:
    if not _ready("scorecard points", "card"):
        return
    card = S["card"]
    dec = decide(card, X_DEV, X_DEC)
    S["decision"] = dec
    print(f"{'input':<24}{'classes':>8}{'fewest':>8}{'most':>7}{'range':>7}   (whole points by class)")
    for j, name in enumerate(FEATURES):
        pts = class_table_points(card, j)
        print(f"{name:<24}{len(pts):>8}{pts.min():>8}{pts.max():>7}{pts.max() - pts.min():>7}")
    exact = OFFSET - FACTOR * (card.coef[0] + np.column_stack(
        [apply_woe(X_DEC[:, j], card.tables[j]) * card.coef[1 + j] for j in range(P)]).sum(axis=1))
    print(f"\nintercept: {intercept_points(card)} points. The whole-point total is within "
          f"{np.abs(dec.total_dec - exact).max():.2f} points of the unrounded score on the decision rows "
          f"(each of the {P + 1} terms is rounded by at most half a point).")
    q = np.quantile(dec.total_dev, [0.0, 0.25, 0.5, 0.75, 1.0]).astype(int)
    print(f"development totals: lowest {q[0]}, quartiles {q[1]}, {q[2]}, {q[3]}, highest {q[4]}.")
    print(f"cut-off: {dec.cut} points, cut so that the lowest {DECLINE_SHARE:.0%} of development applicants are declined "
          f"(strictly fewer than {dec.cut} points); it declines {np.mean(dec.total_dev < dec.cut):.1%} of the "
          f"development rows (ties at the cut are approved) and {dec.declined.mean():.1%} "
          f"({int(dec.declined.sum()):,}) of the decision rows.")
    print(f"Gini of the points on the decision rows: {auc_gini(-dec.total_dec, Y_DEC)[1]:.3f}")


_try("scorecard points", _show_points, needs=("exercise 1",))

## 4. Exercise 2 — `reference_mask()`

A reason says where an applicant fell short *of someone*. Comment 9(b)(2)-5 names two someones:
applicants "at or slightly above the minimum passing score", or all applicants, and says the
average points could be calculated "during the development or use of the system". This lesson
builds both from the development rows, and writes each average down in **whole points**, rounded
by the same rule as every class, as a scorecard document would: so an applicant's shortfall on an
input is a whole number too. "All applicants" is every development row. For "at or slightly above
the minimum passing score" the lesson needs a rule: the band of totals from the cut-off **up to
but not including** the cut-off plus `MARGINAL_WIDTH` points.

<details><summary>💡 Hint 1 — what to think about</summary>

Which end of the band belongs to it? "At" the minimum passing score means an applicant with
exactly the cut-off's points is in the band: that applicant just passes. The other end is a
width above the cut, and a total exactly there is no longer "slightly above".

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Two comparisons joined by AND, one inclusive and one exclusive, evaluated for the whole array at
once. Check the inputs first (a 1-D array without nan, a positive width), and return a boolean
array without changing the one you were given.

</details>

In [ ]:
def reference_mask(total: np.ndarray, cut: float, width: float) -> np.ndarray:
    """Which applicants are "at or slightly above the minimum passing score": cut <= total < cut + width.

    Requirements, each graded:
      * an applicant exactly at the cut-off is IN the band, one exactly `width` above it is NOT;
      * an applicant below the cut-off is never in the band, however close;
      * the result is a boolean array, one entry per applicant, in the order given, and the
        caller's array is not changed;
      * `ValueError` if `total` is not a 1-D array without nan, if `cut` is not a finite number, or
        if `width` is not a positive finite number.

    Example (cut-off 549, a band of 10 points: 549 up to 558):
        >>> reference_mask(np.array([548, 549, 550, 558, 559, 600]), 549, 10).tolist()
        [False, True, True, True, False, False]

    Returns:
        a numpy boolean array.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_reference_mask() -> None:
    got = np.asarray(reference_mask(np.array([548, 549, 550, 558, 559, 600]), 549, 10))
    assert got.tolist() == [False, True, True, True, False, False], (
        f"got {got.tolist()} for the docstring's example; the band is 549 <= total < 559: the cut-off "
        "itself is in, cut + width is out")
    assert got.dtype == bool, "return a boolean array"
    t = np.array([3.0, 3.5, 4.0, 4.5, 5.0, 2.5, 4.99])
    before = t.copy()
    got = np.asarray(reference_mask(t, 3.5, 1.5)).tolist()
    assert got == [False, True, True, True, False, False, True], (
        f"got {got} for totals {t.tolist()} with cut 3.5 and width 1.5: the band is 3.5 <= total < 5.0")
    assert np.array_equal(t, before), "the caller's array was changed"
    for bad in ((np.array([[1, 2]]), 1, 1), (np.array([1.0, np.nan]), 1, 1), (np.array([1.0]), float("inf"), 1),
                (np.array([1.0]), 1, 0), (np.array([1.0]), 1, -3)):
        try:
            reference_mask(*bad)
        except ValueError:
            continue
        raise AssertionError("a 2-D array, a nan total, a cut that is not finite, or a width that is not positive "
                             "must raise ValueError")
    print("exercise 2 looks right")

In [ ]:
_try("exercise 2", _check_reference_mask)

In [ ]:
def whole_reference(points: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """The reference's average points on each input, rounded to whole points by the rule of every class. Given."""
    return np.floor(points[mask].mean(axis=0) + 0.5).astype(np.int64)


def _show_references() -> None:
    dec = S["decision"]
    marginal = reference_mask(dec.total_dev, dec.cut, MARGINAL_WIDTH)
    refs = {"all applicants": np.ones(N_DEV, dtype=bool), "at or just above the cut-off": marginal}
    S["refs"] = {name: mask for name, mask in refs.items()}
    S["ref_means"] = {name: whole_reference(dec.points_dev, mask) for name, mask in refs.items()}
    S["ref_exact"] = {name: dec.points_dev[mask].mean(axis=0) for name, mask in refs.items()}
    print(f"{'reference population':<30}{'rows':>8}{'share':>8}   average points of each input, in whole points")
    print(f"{'':<46}" + "".join(f"{name:>12}" for name in SHORT))
    for name, mask in refs.items():
        print(f"{name:<30}{int(mask.sum()):>8,}{mask.mean():>8.1%}   " + "".join(f"{v:>12d}" for v in S["ref_means"][name]))
    gap = S["ref_means"]["all applicants"] - S["ref_means"]["at or just above the cut-off"]
    print("\naverage points of the 'all applicants' reference minus the other, by input: "
          + ", ".join(f"{FEATURES[j]} {gap[j]:+d}" for j in range(P)))
    print("An applicant's shortfall on an input is the reference's average points minus the applicant's. Where the two")
    print("references' averages are far apart, the choice of reference can change which input an applicant falls short on.")


_try("references", _show_references, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — `principal_reasons()`

For one declined applicant, the **shortfall** on an input is how much that input cost the
applicant against the reference: the reference's mean points minus the applicant's points, so a
positive number means the input pulled the applicant down. The principal reasons are the inputs
with the largest shortfalls. Three rules turn that sentence into code you can regenerate:

- only a **strictly positive** shortfall can be a reason: an input where the applicant is ahead
  of the reference, or level with it, did not cost anything;
- when two shortfalls are **equal**, the lower input number comes first (a notice that cannot be
  reproduced cannot be audited, so a tie is never left to the sort routine's mood);
- when fewer than `k` inputs have a positive shortfall, the places that are left hold
  `NO_REASON` (-1), after the real ones.

The rule is written for any matrix of shortfalls, so it serves the scorecard now and the
Shapley attributions of a boosted model later.

<details><summary>💡 Hint 1 — what to think about</summary>

Sorting is the easy part; the edges are not. What does "largest" mean for a row where every entry
is zero or negative? What should happen to the entries after the last positive one? And a default
`np.argsort` is allowed to put equal values in any order: which keyword makes it keep the lower
input number first? Note that sorting the *negated* values keeps the order of ties.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the shape and the value of `k`. Order each row from the largest shortfall to the
smallest with a stable sort, keep the first `k` positions, and look up the shortfall at each of
them. Wherever that shortfall is not greater than zero, replace the input number with
`NO_REASON`. If `k` is larger than the number of inputs, add the missing places as `NO_REASON`.

</details>

In [ ]:
def principal_reasons(shortfall: np.ndarray, k: int) -> np.ndarray:
    """The k principal reasons of each applicant: the inputs with the largest strictly positive shortfall.

    Requirements, each graded:
      * row i of the result lists, best first, the (at most) k inputs with the largest shortfall in
        row i of `shortfall`, counting only shortfalls GREATER THAN ZERO (zero and negative never);
      * equal shortfalls: the lower input number comes first;
      * if fewer than k inputs have a positive shortfall, the places left over hold NO_REASON (-1),
        after the real reasons; k may be larger than the number of inputs;
      * the result is an integer array of shape (n, k), and the caller's array is not changed;
      * `ValueError` if `shortfall` is not a 2-D array of numbers without nan, or if `k` is not a
        whole number of at least 1.

    Example (three applicants, four inputs; the second is ahead everywhere, the third ties 1 and 3):
        >>> principal_reasons(np.array([[0, 5, 2, 0], [-3, 0, -1, 0], [0, 4, 0, 4]]), 2).tolist()
        [[1, 2], [-1, -1], [1, 3]]

    Returns:
        a numpy int64 array.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_principal_reasons() -> None:
    sf = np.array([[0, 5, 2, 0], [-3, 0, -1, 0], [0, 4, 0, 4]])
    got = np.asarray(principal_reasons(sf, 2)).tolist()
    assert got == [[1, 2], [-1, -1], [1, 3]], (
        f"got {got} for the docstring's example, expected [[1, 2], [-1, -1], [1, 3]]: only a shortfall greater "
        "than zero is a reason (row 2 has none), and a tie goes to the lower input number (row 3)")
    wide = np.asarray(principal_reasons(np.array([[1.5, 0.0, 3.0]]), 5)).tolist()
    assert wide == [[2, 0, -1, -1, -1]], (
        f"got {wide} for k = 5 on three inputs, expected [[2, 0, -1, -1, -1]]: pad the places that are left with -1")
    ties = np.zeros((1, 12))
    ties[0, [3, 5, 8, 11]] = 2.0
    got = np.asarray(principal_reasons(ties, 4)).tolist()
    assert got == [[3, 5, 8, 11]], (
        f"four equal shortfalls came back as {got}; equal shortfalls keep their input order (ask the sort to be stable)")
    sf2 = np.array([[0.0, 0.0, 0.5], [2.0, 1.0, 0.0]])
    before = sf2.copy()
    out = np.asarray(principal_reasons(sf2, 1))
    assert out.tolist() == [[2], [0]] and np.issubdtype(out.dtype, np.integer), (
        f"got {out.tolist()} (dtype {out.dtype}) for k = 1, expected [[2], [0]] as integers")
    assert np.array_equal(sf2, before), "the caller's array was changed"
    for bad in ((np.array([1.0, 2.0]), 1), (np.array([[1.0, np.nan]]), 1), (np.array([[1.0]]), 0), (np.array([[1.0]]), 1.5)):
        try:
            principal_reasons(*bad)
        except ValueError:
            continue
        raise AssertionError("a 1-D array, a nan, a k below 1 and a k that is not a whole number must raise ValueError")
    print("exercise 3 looks right")

In [ ]:
_try("exercise 3", _check_principal_reasons)

Now the points method of the commentary, for every declined applicant, against each reference.
The shortfall on each input is the reference's average points minus the applicant's, both in whole
points, and the first `MAX_REASONS` reasons are kept so that later cells can show a shorter list by
cutting columns off the right. The cell also counts the applicants for whom a **tie** among the
largest shortfalls decides the notice (two equal shortfalls among the top `NOTICE_REASONS + 1`: they
decide the order of the reasons, or which of two equal inputs takes the last place), which is where
your tie rule matters, and writes one notice the way a lender should and one the way the
regulation says is not enough.

In [ ]:
def render_notice(reasons_row: np.ndarray, places: int = NOTICE_REASONS) -> str:
    """The reasons of one applicant as the notice would state them, `places` of them. Given."""
    stated = [REASON_TEXT[int(c)] for c in list(reasons_row)[:places] if int(c) != NO_REASON]
    if not stated:
        return "(no principal reason: the applicant is not behind the reference on any input)"
    return "; ".join(f"{i}. {text}" for i, text in enumerate(stated, 1))


def _show_card_reasons() -> None:
    if not _ready("card reasons", "decision", "ref_means"):
        return
    dec = S["decision"]
    points = dec.points_dec[dec.declined]
    S["card_short"], S["card_reasons"] = {}, {}
    for name, mean in S["ref_means"].items():
        S["card_short"][name] = mean - points
        S["card_reasons"][name] = principal_reasons(S["card_short"][name], MAX_REASONS)
    n = len(points)
    print(f"{int(dec.declined.sum()):,} declined applicants on the decision rows. Share whose FIRST reason is:")
    print(f"{'':<30}" + "".join(f"{name:>12}" for name in SHORT) + f"{'no reason':>12}")
    for name, r in S["card_reasons"].items():
        counts = np.bincount(r[:, 0] + 1, minlength=P + 1)
        print(f"{name:<30}" + "".join(f"{counts[c + 1] / n:>12.1%}" for c in range(P)) + f"{counts[0] / n:>12.1%}")
    for name, r in S["card_reasons"].items():
        have = (r != NO_REASON).sum(axis=1)
        top = np.sort(S["card_short"][name], axis=1)[:, ::-1][:, :NOTICE_REASONS + 1]       # the largest shortfalls, largest first
        tied = float(np.mean(((top[:, :-1] == top[:, 1:]) & (top[:, :-1] > 0)).any(axis=1)))
        print(f"{name:<30} reasons per applicant: {np.mean(have):.2f} on average; {(have >= 1).mean():.1%} have at least one, "
              f"{(have >= NOTICE_REASONS).mean():.1%} have {NOTICE_REASONS}; a tie among the top {NOTICE_REASONS + 1} shortfalls decides the notice for {tied:.1%}")
    print("\nAn applicant's shortfalls add up to the reference's total (its average points added up) minus the applicant's")
    print("total, which is positive whenever the applicant is below the reference's total: that is why nobody should be left")
    print("without a reason while the cut-off sits below both references' totals, and the counts above are the check.")
    ref = "at or just above the cut-off"
    for i in (0, 1):
        r = S["card_reasons"][ref][i]
        print(f"\ndeclined applicant {i + 1}: {int(dec.total_dec[dec.declined][i])} points against a cut-off of {dec.cut}; reference: {ref}")
        print("  inputs' points:  " + ", ".join(f"{FEATURES[j]} {points[i, j]:+d}" for j in range(P)))
        print("  notice:          " + render_notice(r))
    print("\nWhat the regulation calls insufficient: 'Your application was declined because you did not reach the")
    print("qualifying score.' That sentence is true of every declined applicant and names no reason.")


_try("card reasons", _show_card_reasons, needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4 — `reasons_differ()`

Two reason arrays for the same applicants (against two references, or before and after a
nudge) can differ in three ways that matter differently to a notice. The **first** reason
changed. The **ordered** list changed in any place. The **set** of reasons changed, whatever
their order. A notice that states reasons without ranking them only changes in the third sense,
and a lender that says "our principal reason" changes in the first.

<details><summary>💡 Hint 1 — what to think about</summary>

Decide what each kind of change ignores. The first reason ignores everything after place one.
The set ignores the order, and it must ignore the `NO_REASON` padding too: two applicants who
each have one reason have the same set. Padding only ever sits at the right.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check that the two arrays are 2-D, of the same shape, with at least one column, and that `how`
is one of the three names. "First" compares the first column, "ordered" compares every column
and asks whether any row differs, and for "set" one way is to sort each row of both arrays
first, so that order stops mattering, and then compare as for "ordered".

</details>

In [ ]:
def reasons_differ(a: np.ndarray, b: np.ndarray, how: str) -> np.ndarray:
    """For each applicant, did the reasons change from row i of `a` to row i of `b`?

    `a` and `b` are reason arrays as principal_reasons returns them: integer, shape (n, k), real
    reasons first and NO_REASON (-1) after them. Requirements, each graded:
      * how="first": True where the FIRST reasons differ (column 0), whatever follows;
      * how="ordered": True where any place differs, order included;
      * how="set": True where the sets of real reasons differ: the same reasons in another order
        are NOT a change, and the NO_REASON padding is not a reason;
      * the result is a boolean array of length n;
      * `ValueError` if the arrays are not 2-D, differ in shape or have no column, or `how` is
        not one of the three names.

    Example (the second applicant swaps the two reasons, the third has a new second reason):
        >>> a = np.array([[1, 2], [1, 2], [1, -1]]); b = np.array([[1, 2], [2, 1], [1, 3]])
        >>> [reasons_differ(a, b, h).tolist() for h in ("first", "ordered", "set")]
        [[False, True, False], [False, True, True], [False, False, True]]

    Returns:
        a numpy boolean array.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_reasons_differ() -> None:
    a = np.array([[1, 2], [1, 2], [1, -1]])
    b = np.array([[1, 2], [2, 1], [1, 3]])
    want = {"first": [False, True, False], "ordered": [False, True, True], "set": [False, False, True]}
    for how, w in want.items():
        got = np.asarray(reasons_differ(a, b, how)).tolist()
        assert got == w, (f"how={how!r} gave {got}, expected {w} for the docstring's example: "
                          "'first' looks at place one only, 'ordered' at every place, 'set' ignores the order")
    pad = np.asarray(reasons_differ(np.array([[3, -1, -1], [4, -1, -1]]), np.array([[3, -1, -1], [4, 5, -1]]), "set")).tolist()
    assert pad == [False, True], (f"got {pad} when one applicant has the same single reason on both sides and "
                                  "the other gains a reason: the padding must not count as a reason or as a difference")
    first = np.asarray(reasons_differ(np.array([[-1, -1], [2, 0]]), np.array([[0, -1], [2, 1]]), "first")).tolist()
    assert first == [True, False], (f"got {first}: a first reason that appears where there was none is a change, "
                                    "and a change after place one is not a change of the first reason")
    for bad in (("first", np.array([[1]]), np.array([[1, 2]])), ("first", np.array([1, 2]), np.array([1, 2])),
                ("rank", a, b), ("set", np.zeros((2, 0), dtype=int), np.zeros((2, 0), dtype=int))):
        how, x, y = bad
        try:
            reasons_differ(x, y, how)
        except ValueError:
            continue
        raise AssertionError("arrays of different shapes, 1-D arrays, no column or an unknown `how` must raise ValueError")
    print("exercise 4 looks right")

In [ ]:
_try("exercise 4", _check_reasons_differ)

In [ ]:
def _share(a: np.ndarray, b: np.ndarray, how: str, k: int = NOTICE_REASONS) -> float:
    """Share of applicants whose reasons (the first `k` places) differ between two arrays. Given; uses your reasons_differ."""
    return float(np.mean(reasons_differ(a[:, :k], b[:, :k], how)))


def _show_card_reference_flips() -> None:
    if not _ready("reference flips", "card_reasons"):
        return
    r_all, r_marg = S["card_reasons"]["all applicants"], S["card_reasons"]["at or just above the cut-off"]
    S["card_ref_flip"] = {how: _share(r_all, r_marg, how) for how in ("first", "ordered", "set")}
    print(f"scorecard: share of the {len(r_all):,} declined applicants whose reasons (the first {NOTICE_REASONS} places) change")
    print("when ONLY the reference population changes, from 'all applicants' to 'at or just above the cut-off':")
    for how, v in S["card_ref_flip"].items():
        print(f"  {how:<8}{v:>8.1%}")
    print(f"\nBoth references follow methods the commentary describes, and they give a different list of reasons in order to "
          f"{S['card_ref_flip']['ordered']:.1%} of declined applicants.")
    print("That is one book: the replicate books near the end show how much this share moves from book to book.")
    print("The regulation does not say which to prefer, so the choice has to be written down and kept.")


_try("reference flips", _show_card_reference_flips, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 7. Exercise 5 — `code_shortfall()`: a reason code is not an input

A notice states **reason codes**, and a code is a cause, not a column. Two columns can describe
one cause: a second credit bureau's reading of utilisation moves with the first and carries
nothing new. A scorecard fitted on both **splits the weight** between them, so each column's
shortfall is smaller than the cause's, and a list of reasons built input by input can name the
cause twice (using up a place on the notice) or rank something smaller above it.

The fix is to add up the shortfalls of the inputs that share a code before choosing reasons. For
an additive score that is exact: the code's shortfall is the points the cause cost the applicant.

<details><summary>💡 Hint 1 — what to think about</summary>

`code_of[j]` is the code of input `j`; several inputs can share a code and a code can have no
input at all (its column is then all zeros). What dtype should the sums have? An array of whole
points should stay whole, so that ties stay exact. And what must happen to the caller's array?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate that `code_of` is an integer array with one entry per input (floats are refused, whole
values or not) and that every entry is a code in range, a negative one included. Make an
output array of zeros with one column per code and the dtype of `shortfall`. For each input, add
its column into the column of its code (`np.add.at` does that in one call even when several
inputs share a code; plain fancy-index assignment keeps only the last one).

</details>

In [ ]:
def code_shortfall(shortfall: np.ndarray, code_of: np.ndarray, n_codes: int) -> np.ndarray:
    """Add up the shortfalls of the inputs that share a reason code.

    Requirements, each graded:
      * column c of the result is the SUM of the columns j of `shortfall` with code_of[j] == c;
      * a code with no input is a column of zeros;
      * the result has shape (n, n_codes) and the dtype of `shortfall` (whole points stay whole);
      * the caller's arrays are not changed;
      * `ValueError` if `shortfall` is not 2-D, if `code_of` is not an integer-typed 1-D array with one
        entry per input (an array of floats is refused even when its values are whole), if `n_codes`
        is below 1, or if a code is outside 0 .. n_codes - 1 (a negative code is refused: it is not read
        as counting from the end).

    Example (inputs 0 and 2 share code 0; code 2 has no input):
        >>> code_shortfall(np.array([[3, 4, 5], [1, 0, -2]]), np.array([0, 1, 0]), 3).tolist()
        [[8, 4, 0], [-1, 0, 0]]

    Returns:
        a numpy array of shape (n, n_codes).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_code_shortfall() -> None:
    sf = np.array([[3, 4, 5], [1, 0, -2]])
    got = np.asarray(code_shortfall(sf, np.array([0, 1, 0]), 3))
    assert got.tolist() == [[8, 4, 0], [-1, 0, 0]], (
        f"got {got.tolist()} for the docstring's example, expected [[8, 4, 0], [-1, 0, 0]]: the shortfalls of the "
        "inputs that share a code are ADDED (not averaged, and not only the last one kept)")
    assert np.issubdtype(got.dtype, np.integer), "whole points must stay whole: give the result the dtype of the input"
    f = np.asarray(code_shortfall(np.array([[0.5, 1.25, -0.75, 2.0]]), np.array([1, 0, 1, 1]), 2))
    assert f.tolist() == [[1.25, 1.75]], f"got {f.tolist()}, expected [[1.25, 1.75]] for a float input"
    before = sf.copy()
    code_shortfall(sf, np.array([0, 1, 0]), 3)
    assert np.array_equal(sf, before), "the caller's array was changed"
    for bad in ((sf, np.array([0, 1]), 3), (sf, np.array([0, 1, 3]), 3), (sf, np.array([0, -1, 2]), 3), (sf[0], np.array([0]), 1),
                (sf, np.array([0, 1, 0]), 0), (sf, np.array([0.0, 1.0, 0.0]), 3)):
        try:
            code_shortfall(*bad)
        except ValueError:
            continue
        raise AssertionError("a code_of of the wrong length or of floats, a code out of range (a negative one too), a 1-D "
                             "shortfall or n_codes below 1 must raise ValueError")
    print("exercise 5 looks right")

In [ ]:
_try("exercise 5", _check_code_shortfall)

In [ ]:
def _show_twin() -> None:
    """A second reading of utilisation, refit, and the notice built by input and by code."""
    if not _ready("twin", "card_reasons", "card"):
        return
    rng = np.random.default_rng(SEED + 1)
    twin = np.clip(BOOK.X[:, 0] + 0.02 * rng.standard_normal(len(BOOK.X)), 0.0, 1.3)
    X6 = np.column_stack([BOOK.X, twin])
    card6 = fit_scorecard(X6[DEV], Y_DEV, np.append(DIRECTIONS, 1))
    dec6 = decide(card6, X6[DEV], X6[DEC])
    short6 = whole_reference(dec6.points_dev, np.ones(N_DEV, dtype=bool)) - dec6.points_dec[dec6.declined]
    codes = np.array([0, 1, 2, 3, 4, 0])              # the twin is a second reading of utilisation: code 0
    by_input = principal_reasons(short6, NOTICE_REASONS)
    by_code = principal_reasons(code_shortfall(short6, codes, P), NOTICE_REASONS)
    as_code = np.where(by_input >= 0, codes[np.maximum(by_input, 0)], NO_REASON)
    base = S["card_reasons"]["all applicants"][:, :NOTICE_REASONS]
    card = S["card"]
    names_twice = lambda r, cols: float(np.mean(np.isin(r, cols).sum(axis=1) > 1))      # noqa: E731  the cause fills both places
    util_first = lambda r, cols: float(np.mean(np.isin(r[:, 0], cols)))      # noqa: E731
    print(f"correlation of the two readings of utilisation: {np.corrcoef(BOOK.X[DEV, 0], twin[DEV])[0, 1]:.4f}")
    print(f"coefficient on utilisation: {card.coef[1]:.3f} alone; {card6.coef[1]:.3f} and {card6.coef[6]:.3f} once the twin is in")
    print(f"\n{int(dec6.declined.sum()):,} declined applicants (the twin model has its own cut-off); the reference is all applicants\n")
    print(f"{'':<34}{'utilisation first':>19}{'names it twice':>16}")
    print(f"{'without the twin':<34}{util_first(base, [0]):>19.1%}{'-':>16}")
    print(f"{'with the twin, by input':<34}{util_first(by_input, [0, 5]):>19.1%}{names_twice(by_input, [0, 5]):>16.1%}")
    print(f"{'with the twin, by reason code':<34}{util_first(by_code, [0]):>19.1%}{names_twice(by_code, [0]):>16.1%}")
    print(f"\nthe first CAUSE differs between the by-input and by-code lists for "
          f"{np.mean(reasons_differ(as_code[:, :1], by_code[:, :1], 'first')):.1%} of declined applicants.")
    print("A two-place notice that names utilisation twice has used a place on one cause. Summing first costs")
    print("nothing for an additive score, and it ranks the cause itself.")


_try("twin", _show_twin, needs=("exercise 1", "exercise 3", "exercise 4", "exercise 5"))

## 8. Exercise 6 — `perturbation_flip_rate()`: does the reason survive a hair's breadth?

A reason on a notice should not depend on a recording detail. If an applicant's utilisation had
been 41.3% instead of 41.4%, the principal reason should be the same. A scorecard is a step
function of each input, so a tiny move across a class edge changes that input's points by a whole
class step; a boosted model is a step function with far more steps. The test asks, for every
declined applicant, whether the reasons change when **any one continuous input** is moved by a
negligible amount, in **either direction**.

The size of a move is `eps` times the input's own **spread** (`scale`, here the development
standard deviation), so one number serves inputs measured in different units. Only the columns
you name are moved, one at a time. The two counts are left alone: a count of delinquencies is
recorded exactly, and moving it by a fraction is not a small recording error but an input that
cannot exist.

<details><summary>💡 Hint 1 — what to think about</summary>

A step function changes at an edge, and an applicant can sit just below it or just above it.
What does that say about moving in one direction only? The reasons you compare against are the
ones for the *original* values. And what happens to your copy of `X` if it holds whole numbers
and you add a small fraction to a column?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the arguments first: `X` a non-empty 2-D array, every column in range (a negative one is
not), `scale` one entry per column, `eps` finite and not negative, and `how` one of the three
names, whether or not there is a column to move. Work on a float copy of `X`. Ask `reasons_of`
for the reasons of the unchanged rows once. Then, for each named column and for each of the two
directions, make a fresh copy, add `direction * eps * scale[column]` to that one column, ask for
the reasons, compare with `reasons_differ` and remember every applicant who differed in any of
the moves. An applicant is flipped if at least one move changed their reasons.

</details>

In [ ]:
def perturbation_flip_rate(reasons_of: Callable[[np.ndarray], np.ndarray], X: np.ndarray, columns: Any,
                           scale: np.ndarray, eps: float, how: str) -> tuple[float, np.ndarray]:
    """The share of applicants whose reasons change when one input moves by eps * scale, up or down.

    `reasons_of(X)` returns the reason array (shape (n, k), as principal_reasons returns it) for the
    rows of `X`. Requirements, each graded:
      * the base reasons are those of the unchanged rows; then, for each DISTINCT column in
        `columns` and each of the two directions (-1 and +1), one copy of the rows is made with
        direction * eps * scale[column] added to that column only, and `reasons_of` is called on it:
        1 + 2 * (number of distinct columns) calls in all;
      * an applicant is flipped if `reasons_differ(base, moved, how)` is True for ANY of those moves;
      * `X` is read as floats (a whole-number array must still move by a fraction) and is not changed,
        and every call of `reasons_of` gets an array of its own, so a reason function that changes its
        argument cannot disturb the next move;
      * the result is (rate, flipped): the share flipped as a Python float, and the boolean array;
      * with no columns nothing moves and nobody is flipped;
      * `ValueError` if `X` is not a non-empty 2-D array, a column lies outside 0 .. (the number of
        columns of `X`) - 1 (a negative column is refused: it is not read as counting from the end),
        `scale` is not a vector with one entry per column of `X`, `eps` is negative or not finite, or
        `how` is not one of the three names `reasons_differ` takes (even when no column is named).

    Example (a step model whose only reason flips at x0 = 1; the first row is just below the edge, the
    second just above it, the third far from it):
        >>> step = lambda A: np.where(A[:, :1] < 1.0, 0, 1)
        >>> rate, flipped = perturbation_flip_rate(step, np.array([[0.995], [1.005], [3.0]]), (0,),
        ...                                        np.array([1.0]), 0.01, "first")
        >>> round(rate, 4), flipped.tolist()
        (0.6667, [True, True, False])

    Returns:
        (rate, flipped).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_perturbation_flip_rate() -> None:
    step = lambda A: np.where(A[:, :1] < 1.0, 0, 1)      # noqa: E731  a step in the first column only
    X = np.array([[0.995, 5.0], [1.005, 5.0], [3.0, 5.0], [0.2, 0.999]])
    rate, flipped = perturbation_flip_rate(step, X, (0,), np.array([1.0, 1.0]), 0.01, "first")
    assert np.asarray(flipped).tolist() == [True, True, False, False], (
        f"got {np.asarray(flipped).tolist()} for rows just below, just above and far from the edge: a move in "
        "EITHER direction can cross it, so try the move up and the move down, and keep an applicant flipped if "
        "any one of them changed the reasons")
    assert abs(rate - 0.5) < 1e-12 and isinstance(rate, float), f"rate {rate!r}: the share flipped, as a Python float (2 of 4)"
    _, wide = perturbation_flip_rate(step, X, (0,), np.array([2.0, 1.0]), 0.01, "first")
    assert np.asarray(wide).tolist() == [True, True, False, False], "the move is eps TIMES THE SCALE of that column"
    _, other = perturbation_flip_rate(step, np.array([[0.995, 0.0], [1.005, 0.0]]), (1,), np.array([1.0, 1.0]), 0.01, "first")
    assert not np.asarray(other).any(), "only the columns you are asked to move may move: column 0 was moved too"
    step_half = lambda A: np.where(A[:, :1] < 1.5, 0, 1)      # noqa: E731
    ints = np.array([[2, 0], [1, 0]])
    _, from_ints = perturbation_flip_rate(step_half, ints, (0,), np.array([1.0, 1.0]), 0.01, "first")
    assert np.asarray(from_ints).tolist() == [False, False], (
        f"got {np.asarray(from_ints).tolist()} for whole-number input where no move of 0.01 reaches the edge at 1.5: "
        "a copy that holds whole numbers rounds 1.99 down to 1 and finds a flip that is not there; work on a float copy")
    calls = []

    def spy(A):
        calls.append(A.copy())
        return step(A)

    keep = X.copy()
    perturbation_flip_rate(spy, X, (0, 1, 0), np.array([1.0, 1.0]), 0.01, "first")
    assert len(calls) == 5, f"reasons_of was called {len(calls)} times; with two distinct columns that is 1 + 2 * 2 = 5"
    assert np.array_equal(calls[0], keep) and np.array_equal(X, keep), (
        "the first call must see the unchanged rows, and X must not be changed")
    zero, none = perturbation_flip_rate(step, X, (), np.array([1.0, 1.0]), 0.01, "first")
    assert zero == 0.0 and not np.asarray(none).any(), "no columns: nothing moves, nobody flips"
    for bad in ((step, X[:, 0], (0,), np.array([1.0]), 0.01, "first"), (step, X, (5,), np.array([1.0, 1.0]), 0.01, "first"),
                (step, X, (-1,), np.array([1.0, 1.0]), 0.01, "first"), (step, X, (0,), np.array([1.0]), 0.01, "first"),
                (step, X, (0,), np.array([1.0, 1.0]), -0.1, "first"), (step, X, (0,), np.array([1.0, 1.0]), 0.01, "rank"),
                (step, X, (), np.array([1.0, 1.0]), 0.01, "rank")):
        try:
            perturbation_flip_rate(*bad)
        except ValueError:
            continue
        raise AssertionError("a 1-D X, a column out of range (a negative one too), a scale of the wrong length, a negative eps "
                             "or a `how` other than 'first', 'ordered' and 'set' (with a column to move or without) must raise ValueError")
    print("exercise 6 looks right")

In [ ]:
_try("exercise 6", _check_perturbation_flip_rate, needs=("exercise 4",))

In [ ]:
def card_reasons_of(card: Scorecard, ref_mean: np.ndarray, k: int = MAX_REASONS) -> Callable[[np.ndarray], np.ndarray]:
    """The reasons of a scorecard as a function of raw inputs: the shortfall against the reference's average points, then the top k. Given."""
    return lambda X: principal_reasons(ref_mean - scorecard_points(card, X), k)


def _show_card_perturbation() -> None:
    if not _ready("scorecard perturbation", "decision", "card", "ref_means"):
        return
    dec, card = S["decision"], S["card"]
    X_dec = X_DEC[dec.declined]
    ref = S["ref_means"]["all applicants"]
    reasons_of = card_reasons_of(card, ref, NOTICE_REASONS)
    points = dec.points_dec[dec.declined]
    print(f"scorecard, reference 'all applicants', {len(X_dec):,} declined applicants. A move is eps x the input's")
    print("development standard deviation, up and down, on utilisation, age of account and area index in turn.")
    print(f"{'eps':>8}{'points of some input changed':>31}{'first reason':>14}{'ordered list':>14}{'set of reasons':>16}")
    rows = {}
    for eps in EPS_SWEEP:
        rates = {how: perturbation_flip_rate(reasons_of, X_dec, CONTINUOUS, SD_DEV, eps, how) for how in ("first", "ordered", "set")}
        changed = np.zeros(len(X_dec), dtype=bool)
        for j in CONTINUOUS:
            for direction in (-1.0, 1.0):
                moved = X_dec.copy()
                moved[:, j] += direction * eps * SD_DEV[j]
                changed |= (scorecard_points(card, moved) != points).any(axis=1)
        rows[eps] = {how: r[0] for how, r in rates.items()} | {"changed": float(changed.mean())}
        print(f"{eps:>8g}{rows[eps]['changed']:>31.1%}{rows[eps]['first']:>14.1%}{rows[eps]['ordered']:>14.1%}{rows[eps]['set']:>16.1%}")
    S["card_flip"] = rows
    first_flipped = perturbation_flip_rate(reasons_of, X_dec, CONTINUOUS, SD_DEV, EPS_MAIN, "first")[1]
    sf = np.sort(ref - points, axis=1)
    gap = sf[:, -1] - sf[:, -2]
    crossed = rows[EPS_MAIN]["changed"] * len(X_dec)
    print(f"\nat eps = {EPS_MAIN:g}: of the {int(round(crossed)):,} applicants for whom the move changed some input's points, "
          f"{first_flipped.sum() / max(crossed, 1.0):.1%} got a different first reason.")
    if first_flipped.any() and (~first_flipped).any():
        print(f"\nat eps = {EPS_MAIN:g}: the median gap between the largest and the second-largest shortfall is "
              f"{np.median(gap[first_flipped]):.1f} points for applicants whose first reason flips and "
              f"{np.median(gap[~first_flipped]):.1f} for those whose does not.")
    print("A move can only matter if it crosses a class edge, and it only changes the first reason if the applicant's")
    print("two largest shortfalls are closer than the step it causes.")


_try("scorecard perturbation", _show_card_perturbation,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6"))

## 9. The challenger, and exact attributions for a model that is not a sum

The challenger is the **monotone-constrained gradient-boosted tree model** of `P05-L03`, on the
same points scale as the scorecard (`OFFSET` minus `FACTOR` times its raw score), cut at its own
cut-off by the same rule (the lowest `DECLINE_SHARE` of development applicants are declined). Its
code is given below as a compact copy of that lesson's
worked solution (the same split search, the same bounds handed down after every constrained
split, the same boosting loop, the same choice of the number of trees on the validation rows);
it adds one thing, a **table form of each tree** that returns exactly the same numbers much
faster and that the attributions below exploit.

A boosted model is not a sum of one term per input, so "the applicant's points minus the
reference's mean points" has no meaning for one input. **Exact Shapley attributions** do: the
Shapley value of input `j` for one applicant is `j`'s average marginal contribution over every
order in which the inputs could be revealed, where "revealing" an input means replacing the
background applicants' value of it with this applicant's. `P04-L08` built that enumeration; here
it is given, in a batched form for many applicants at once, with the original per-applicant
functions beside it so the next cell can check that the routes agree. With `P` inputs that is
`2**P` coalitions per applicant, so brute force doubles in cost with every input you add and
multiplies it by the size of the background. The **background** plays the part of the reference
population: its mean score is the baseline the attributions add up from. A tree ensemble has
structure a general model does not: each tree is a table of a modest number of cells, and the
background then only enters as a histogram over those cells, so the background can be the whole
reference population, exactly as it is for the scorecard.

In [ ]:
def grad_hess(raw: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Gradient and hessian of the logistic loss in the raw score (P05-L03, exercise 1). Given."""
    p = sigmoid(raw)
    return p - y, np.maximum(p * (1.0 - p), HESS_FLOOR)


def leaf_value(G: float, H: float, lam: float, lo: float = -math.inf, hi: float = math.inf) -> float:
    """The leaf weight -G / (H + lam), clipped into [lo, hi] (P05-L03, exercise 2). Given."""
    return float(min(max(-G / (H + lam), lo), hi))


def make_edges(X: np.ndarray, n_bins: int = N_BINS) -> list:
    """Per-input inner bin edges: quantiles of the development rows, de-duplicated. Given."""
    out = []
    for j in range(X.shape[1]):
        e = np.unique(np.quantile(X[:, j], np.linspace(0.0, 1.0, n_bins + 1)[1:-1]))
        out.append(e[e > X[:, j].min()])
    return out


def to_bins(X: np.ndarray, edges: list) -> np.ndarray:
    """Integer bin of every value: bin b holds edges[b-1] <= x < edges[b]. Given."""
    return np.column_stack([np.searchsorted(e, X[:, j], side="right") for j, e in enumerate(edges)]).astype(np.int64)


class Split(NamedTuple):
    gain: float
    feature: int
    bin: int          # rows with Xb[:, feature] <= bin go left
    w_left: float
    w_right: float


def best_split(Xb, g, h, rows, constraints, n_bins, lam, min_child_hess, min_gain, lo=-math.inf, hi=math.inf):
    """Best valid split of the node made of `rows`, or None (P05-L03, exercise 3). Given.

    A candidate is valid when both children hold a row and at least `min_child_hess` of hessian, and, for
    an input that must not push risk down, the CLIPPED left weight does not exceed the right one.
    """
    if rows.size < 2:
        return None
    gr, hr = g[rows], h[rows]
    G, H = float(gr.sum()), float(hr.sum())
    if not H + lam > 0:
        return None

    def score(Gs, Hs, w):
        return -Gs * w - 0.5 * (Hs + lam) * w * w

    parent = score(G, H, leaf_value(G, H, lam, lo, hi))
    best = None
    for j in range(Xb.shape[1]):
        col = Xb[rows, j]
        GL = np.cumsum(np.bincount(col, weights=gr, minlength=n_bins))[:-1]
        HL = np.cumsum(np.bincount(col, weights=hr, minlength=n_bins))[:-1]
        NL = np.cumsum(np.bincount(col, minlength=n_bins))[:-1]
        GR, HR, NR = G - GL, H - HL, rows.size - NL
        valid = ((NL >= 1) & (NR >= 1) & (HL >= min_child_hess) & (HR >= min_child_hess)
                 & (HL + lam > 0) & (HR + lam > 0))
        if not valid.any():
            continue
        with np.errstate(divide="ignore", invalid="ignore"):
            wl = np.clip(-GL / (HL + lam), lo, hi)
            wr = np.clip(-GR / (HR + lam), lo, hi)
        if constraints[j] > 0:
            valid &= wl <= wr
        elif constraints[j] < 0:
            valid &= wl >= wr
        if not valid.any():
            continue
        gain = np.where(valid, score(GL, HL, wl) + score(GR, HR, wr) - parent, -np.inf)
        b = int(np.argmax(gain))
        if gain[b] > min_gain and (best is None or gain[b] > best.gain):
            best = Split(float(gain[b]), j, b, float(wl[b]), float(wr[b]))
    return best


class Tree(NamedTuple):
    feature: np.ndarray   # int per node: the input split on, or -1 at a leaf
    bin: np.ndarray       # int per node: rows with Xb[:, feature] <= bin go left
    left: np.ndarray
    right: np.ndarray
    value: np.ndarray     # the leaf weight (0.0 at an internal node)


def predict_tree(tree: Tree, Xb: np.ndarray) -> np.ndarray:
    """The weight of the leaf each row of `Xb` lands in. Given."""
    node = np.zeros(len(Xb), dtype=int)
    while True:
        f = tree.feature[node]
        live = np.flatnonzero(f >= 0)
        if live.size == 0:
            return tree.value[node]
        go_left = Xb[live, f[live]] <= tree.bin[node[live]]
        node[live] = np.where(go_left, tree.left[node[live]], tree.right[node[live]])


def grow_tree(Xb, g, h, constraints, max_depth, n_bins, lam, min_child_hess, min_gain) -> Tree:
    """One depth-limited tree; after a split on a constrained input the children inherit bounds that
    meet at the midpoint of the two clipped child weights (P05-L03, exercise 4). Given."""
    nodes: list = []

    def build(rows, depth, lo, hi):
        index = len(nodes)
        nodes.append(None)
        split = None
        if depth < max_depth:
            split = best_split(Xb, g, h, rows, constraints, n_bins, lam, min_child_hess, min_gain, lo, hi)
        if split is None:
            nodes[index] = (-1, -1, -1, -1, leaf_value(float(g[rows].sum()), float(h[rows].sum()), lam, lo, hi))
            return index
        goes_left = Xb[rows, split.feature] <= split.bin
        c = int(constraints[split.feature])
        l_lo, l_hi, r_lo, r_hi = lo, hi, lo, hi
        if c != 0:
            mid = 0.5 * (split.w_left + split.w_right)
            if c > 0:
                l_hi, r_lo = mid, mid
            else:
                l_lo, r_hi = mid, mid
        left = build(rows[goes_left], depth + 1, l_lo, l_hi)
        right = build(rows[~goes_left], depth + 1, r_lo, r_hi)
        nodes[index] = (split.feature, split.bin, left, right, 0.0)
        return index

    build(np.arange(len(Xb)), 0, -math.inf, math.inf)
    cols = list(zip(*nodes))
    return Tree(np.array(cols[0], dtype=int), np.array(cols[1], dtype=int), np.array(cols[2], dtype=int),
                np.array(cols[3], dtype=int), np.array(cols[4], dtype=float))


class Booster(NamedTuple):
    base: float
    learning_rate: float
    trees: list


def predict_raw(booster: Booster, Xb: np.ndarray, n_trees: int | None = None) -> np.ndarray:
    """Raw score (log-odds of default) of each row using the first `n_trees` trees. Given."""
    out = np.full(len(Xb), booster.base)
    for tree in booster.trees[:len(booster.trees) if n_trees is None else n_trees]:
        out = out + booster.learning_rate * predict_tree(tree, Xb)
    return out


def fit_booster(Xb, y, constraints, n_rounds, learning_rate=LEARNING_RATE, max_depth=MAX_DEPTH, n_bins=N_BINS,
                lam=LAMBDA, min_child_hess=MIN_CHILD_HESS, min_gain=0.0) -> Booster:
    """Boosting from the base score ln(p / (1 - p)), one tree per round (P05-L03, exercise 5). Given."""
    t = np.asarray(y)
    p = float(t.mean())
    base = math.log(p / (1.0 - p))
    raw = np.full(len(t), base)
    trees = []
    for _ in range(n_rounds):
        g, h = grad_hess(raw, t)
        tree = grow_tree(Xb, g, h, constraints, max_depth, n_bins, lam, min_child_hess, min_gain)
        raw = raw + learning_rate * predict_tree(tree, Xb)
        trees.append(tree)
    return Booster(base, float(learning_rate), trees)


def staged_log_loss(booster: Booster, Xb: np.ndarray, y: np.ndarray) -> tuple[int, np.ndarray]:
    """Mean log loss after 0, 1, ..., T trees and the number of trees where it is lowest (P05-L03, exercise 6). Given."""
    raw = np.full(len(y), booster.base)
    losses = [float(np.mean(np.logaddexp(0.0, raw) - y * raw))]
    for tree in booster.trees:
        raw = raw + booster.learning_rate * predict_tree(tree, Xb)
        losses.append(float(np.mean(np.logaddexp(0.0, raw) - y * raw)))
    out = np.array(losses)
    return int(np.argmin(out)), out


class TreeTable(NamedTuple):
    used: tuple         # the inputs this tree splits on, in order
    thresholds: tuple   # for each used input, the sorted distinct bins the tree splits it at
    table: np.ndarray   # the tree's leaf weight in every cell, one axis per used input


class Challenger(NamedTuple):
    booster: Booster      # trimmed to the number of trees the validation rows chose
    edges: list           # the bin edges the trees were grown on (development rows only)
    tables: list          # one TreeTable per tree
    n_inputs: int


def compile_tree(tree: Tree, n_inputs: int) -> TreeTable:
    """A tree as a small table. Given.

    A tree depends on an input only through which side of the tree's few thresholds on that input a
    row falls, so with at most a handful of thresholds per input the whole tree is a table of a few
    dozen cells, found here by asking the tree itself for its weight in each cell. A row's cell on an
    input is the number of that input's thresholds below its bin.
    """
    used = tuple(sorted({int(f) for f in tree.feature if f >= 0}))
    if not used:
        return TreeTable((), (), np.array(float(tree.value[0])))
    thresholds = tuple(np.unique(tree.bin[tree.feature == f]) for f in used)
    shape = tuple(len(t) + 1 for t in thresholds)
    grids = np.meshgrid(*[np.arange(size) for size in shape], indexing="ij")
    probe = np.zeros((int(np.prod(shape)), n_inputs), dtype=np.int64)
    for f, t, grid in zip(used, thresholds, grids):
        probe[:, f] = np.append(t, t[-1] + 1)[grid.ravel()]       # a bin inside each cell
    return TreeTable(used, thresholds, predict_tree(tree, probe).reshape(shape))


def cells_of(tt: TreeTable, Xb: np.ndarray) -> list:
    """For each input the tree uses, the cell (an integer array) each row of binned inputs falls in. Given."""
    return [np.searchsorted(t, Xb[:, f], side="left") for f, t in zip(tt.used, tt.thresholds)]


def lookup_raw(model: Challenger, Xb: np.ndarray) -> np.ndarray:
    """The raw score from the tree tables: the same numbers as predict_raw, several times faster. Given."""
    out = np.full(len(Xb), model.booster.base)
    for tt in model.tables:
        out = out + model.booster.learning_rate * (tt.table[tuple(cells_of(tt, Xb))] if tt.used else float(tt.table))
    return out


def fit_challenger(X_dev, y_dev, X_val, y_val, constraints) -> Challenger:
    """Fit on the development rows, keep the number of trees that is best on the validation rows. Given.

    If the validation loss is still lowest at the last tree, the cap was too low for this book and the
    model is refitted with twice the rounds, up to HARD_ROUNDS, so the answer never depends on the cap.
    """
    edges = make_edges(X_dev)
    Xb_dev, Xb_val = to_bins(X_dev, edges), to_bins(X_val, edges)
    rounds = MAX_ROUNDS
    while True:
        booster = fit_booster(Xb_dev, y_dev, constraints, rounds)
        best, _ = staged_log_loss(booster, Xb_val, y_val)
        if best < rounds or rounds >= HARD_ROUNDS:
            break
        rounds = min(2 * rounds, HARD_ROUNDS)
    booster = booster._replace(trees=booster.trees[:best])
    n_inputs = X_dev.shape[1]
    return Challenger(booster, edges, [compile_tree(t, n_inputs) for t in booster.trees], n_inputs)


def points_from_bins(model: Challenger, Xb: np.ndarray) -> np.ndarray:
    """The challenger's score on the points scale, from binned inputs. Given."""
    return OFFSET - FACTOR * lookup_raw(model, Xb)


def points_from_raw(model: Challenger, X: np.ndarray) -> np.ndarray:
    """The challenger's score on the points scale, from raw inputs. Given."""
    return points_from_bins(model, to_bins(X, model.edges))

The attributions. `shapley_exact` is the enumeration of `P04-L08`, written for many applicants at
once; `coalition_value` and `shapley_values` below it are that lesson's worked solution, one
applicant at a time, kept so the next cell can compare them.

In [ ]:
def shapley_from_values(values: np.ndarray) -> np.ndarray:
    """Shapley values from the value of every coalition. Given.

    `values[i, S]` is the worth to applicant i of the coalition whose members are the set bits of the
    integer S. Each input's value is the sum, over the coalitions S that leave it out, of the weight
    |S|! (p-|S|-1)! / p! times the gain from adding it.
    """
    n, size = values.shape
    p = size.bit_length() - 1
    fact = [math.factorial(k) for k in range(p + 1)]
    phi = np.zeros((n, p))
    for i in range(p):
        bit = 1 << i
        for mask in range(size):
            if mask & bit:
                continue
            phi[:, i] += fact[mask.bit_count()] * fact[p - mask.bit_count() - 1] / fact[p] * (values[:, mask | bit] - values[:, mask])
    return phi


def shapley_exact(value_of_rows: Callable[[np.ndarray], np.ndarray], applicants: np.ndarray,
                  background: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Exact interventional Shapley values for many applicants at once, by brute force. Given (P04-L08, batched).

    For every coalition S of inputs, the value is the mean over the background rows of the score of
    the row with the columns in S replaced by the applicant's values, so the empty coalition is the
    background's mean score and the full one is the applicant's own. Works for any model, at a cost
    of 2**p scorings of (applicants x background) rows.
    Returns (phi of shape (n, p), baseline value, applicant's own score).
    """
    n, p = applicants.shape
    m = len(background)
    values = np.empty((n, 1 << p))
    for mask in range(1 << p):
        keep = np.array([(mask >> j) & 1 for j in range(p)], dtype=bool)
        rows = np.broadcast_to(background, (n, m, p)).copy()
        rows[:, :, keep] = applicants[:, None, keep]
        values[:, mask] = value_of_rows(rows.reshape(-1, p)).reshape(n, m).mean(axis=1)
    return shapley_from_values(values), values[:, 0], values[:, -1]


def shapley_trees(model: Challenger, applicants: np.ndarray, background: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The same numbers as shapley_exact for the challenger, without scoring (applicants x background) rows. Given.

    The value of a coalition is a sum over trees. A tree reads an input only through its cell, so for
    a tree and a subset A of the inputs it uses, the mean over the background of the tree's weight
    with A taken from the applicant is a small table over A's cells, built once from the background's
    histogram over the tree's cells. Each applicant then needs one lookup per tree and subset. The
    background enters only through that histogram, so it can be the whole reference population.
    Both arguments are binned inputs. Returns (phi, baseline value, applicant's own score), on the points scale.
    """
    n, m, p = len(applicants), len(background), model.n_inputs
    lr = model.booster.learning_rate
    raw = np.zeros((n, 1 << p))
    for tt in model.tables:
        if not tt.used:
            raw += lr * float(tt.table)
            continue
        u = len(tt.used)
        a_cells, b_cells = cells_of(tt, applicants), cells_of(tt, background)
        hist = np.bincount(np.ravel_multi_index(b_cells, tt.table.shape), minlength=tt.table.size).reshape(tt.table.shape)
        used_mask = sum(1 << f for f in tt.used)
        for subset in range(1 << u):
            from_applicant = tuple(k for k in range(u) if subset >> k & 1)
            from_background = tuple(k for k in range(u) if not subset >> k & 1)
            weights = hist.sum(axis=from_applicant, keepdims=True) if from_applicant else hist
            mean_weight = (tt.table * weights).sum(axis=from_background) / m if from_background else tt.table * (weights / m)
            gain = mean_weight[tuple(a_cells[k] for k in from_applicant)] if from_applicant else float(mean_weight)
            want = sum(1 << tt.used[k] for k in from_applicant)
            for coalition in range(1 << p):
                if coalition & used_mask == want:
                    raw[:, coalition] += lr * gain
    values = OFFSET - FACTOR * (model.booster.base + raw)
    return shapley_from_values(values), values[:, 0], values[:, -1]


def coalition_value(model: Callable[[np.ndarray], np.ndarray], x: np.ndarray,
                    background: np.ndarray) -> Callable[[tuple], float]:
    """P04-L08's interventional value function for ONE applicant `x`. Given (that lesson's exercise 6)."""
    bg = np.array(background, dtype=float, copy=True)
    x = np.array(x, dtype=float, copy=True).ravel()

    def value(coalition: tuple) -> float:
        cols = [int(c) for c in coalition]
        rows = bg.copy()
        if cols:
            rows[:, cols] = x[cols]
        return float(np.mean(model(rows)))

    return value


def shapley_values(value: Callable[[tuple], float], n_features: int) -> np.ndarray:
    """P04-L08's exact Shapley values of one game, by full enumeration. Given (that lesson's exercise 7)."""
    p = int(n_features)
    worth = {}
    for mask in range(1 << p):
        worth[mask] = float(value(tuple(i for i in range(p) if mask >> i & 1)))
    fact = [math.factorial(k) for k in range(p + 1)]
    phi = np.zeros(p)
    for i in range(p):
        bit = 1 << i
        total = 0.0
        for mask in range(1 << p):
            if mask & bit:
                continue
            size = mask.bit_count()
            total += fact[size] * fact[p - size - 1] / fact[p] * (worth[mask | bit] - worth[mask])
        phi[i] = total
    return phi


def _fit_challenger() -> None:
    model = fit_challenger(X_DEV, Y_DEV, BOOK.X[VAL], BOOK.y[VAL], DIRECTIONS)
    p_dev, p_dec = points_from_raw(model, X_DEV), points_from_raw(model, X_DEC)
    cut = cut_from_development(p_dev, DECLINE_SHARE)
    S["challenger"] = {"model": model, "points_dev": p_dev, "points_dec": p_dec, "cut": cut, "declined": p_dec < cut}
    cells = [t.table.size for t in model.tables]
    print(f"challenger: {len(model.booster.trees)} trees of depth {MAX_DEPTH}, chosen on the validation rows; as tables, "
          f"the trees have {min(cells)} to {max(cells)} cells each.")
    if not _ready("challenger", "decision"):
        return
    dec = S["decision"]
    print(f"\n{'':<24}{'Gini':>8}{'declined':>10}{'group 0':>9}{'group 1':>9}   (decision rows; the planted probability scores "
          f"{auc_gini(BOOK.pd_true[DEC], Y_DEC)[1]:.3f})")
    for name, s, c in (("scorecard", dec.total_dec, dec.cut), ("challenger", p_dec, cut)):
        d = s < c
        print(f"{name:<24}{auc_gini(-s, Y_DEC)[1]:>8.3f}{d.mean():>10.1%}{d[G_DEC == 0].mean():>9.1%}{d[G_DEC == 1].mean():>9.1%}")
    gaps = {name: float((s < c)[G_DEC == 1].mean() - (s < c)[G_DEC == 0].mean()) for name, s, c in
            (("scorecard", dec.total_dec, dec.cut), ("challenger", p_dec, cut))}
    planted = float(BOOK.pd_true[DEC][G_DEC == 1].mean() - BOOK.pd_true[DEC][G_DEC == 0].mean())
    print(f"\ndecline rate of group 1 minus group 0, in percentage points: scorecard {100 * gaps['scorecard']:+.1f}, "
          f"challenger {100 * gaps['challenger']:+.1f}; the planted default probability differs by {100 * planted:+.1f}.")
    print("The rest of the lesson asks what the reasons say about a gap like that.")


_try("challenger", _fit_challenger, needs=("exercise 1",), heavy=True)


def _check_attributions() -> None:
    if not _ready("attributions", "challenger"):
        return
    ch = S["challenger"]
    model = ch["model"]
    idx = np.flatnonzero(ch["declined"])[:3]
    bg_rows = systematic_sample(np.arange(N_DEV), 60)
    bg_raw, apps_raw = X_DEV[bg_rows], X_DEC[idx]
    apps_bins, bg_bins = to_bins(apps_raw, model.edges), to_bins(bg_raw, model.edges)
    single = np.array([shapley_values(coalition_value(lambda A: points_from_raw(model, A), x, bg_raw), P) for x in apps_raw])
    brute, base, own = shapley_exact(lambda A: points_from_bins(model, A), apps_bins, bg_bins)
    tree, base_t, own_t = shapley_trees(model, apps_bins, bg_bins)
    print(f"three declined applicants, a {len(bg_rows)}-row background, three ways to the same attributions:")
    print(f"  P04-L08's per-applicant functions against the batched enumeration, within 1e-9: {bool(np.abs(single - brute).max() < 1e-9)}")
    print(f"  the batched enumeration against the tree tables, within 1e-9:                  {bool(np.abs(brute - tree).max() < 1e-9)}")
    print(f"  attributions add up to the applicant's score minus the background's mean, within 1e-9: "
          f"{bool(np.abs(tree.sum(axis=1) - (own_t - base_t)).max() < 1e-9)}")
    walk = OFFSET - FACTOR * predict_raw(model.booster, to_bins(X_DEC[:2000], model.edges))
    print(f"  and the table scorer returns exactly the same points as walking the trees: "
          f"{bool(np.array_equal(walk, ch['points_dec'][:2000]))}")


_try("attributions", _check_attributions, needs=("exercise 1",), heavy=True)


def _show_identity() -> None:
    if not _ready("identity", "decision", "card", "ref_exact", "card_reasons"):
        return
    dec, card = S["decision"], S["card"]
    base = intercept_points(card)
    idx = np.flatnonzero(dec.declined)[:5]
    apps = X_DEC[idx]
    value = lambda A: base + scorecard_points(card, A).sum(axis=1)                  # noqa: E731
    phi, v0, v_own = shapley_exact(value, apps, X_DEV)
    shortfall = S["ref_exact"]["all applicants"] - dec.points_dec[idx]
    print(f"For the additive scorecard, with the {N_DEV:,} development rows as the background, is the points method")
    print("(the reference's mean points minus the applicant's, input by input) the same thing as exact Shapley?")
    print(f"  largest difference between the two, over {len(idx)} applicants and {P} inputs, below 1e-9: "
          f"{bool(np.abs(phi + shortfall).max() < 1e-9)}")
    print("The reason is arithmetic: for a sum of one term per input, an input's Shapley value is its own term minus the")
    print("mean of that term over the background. The commentary's method is then the Shapley method for an additive")
    print("score, and exact Shapley carries the same idea to a model that is not additive.")
    declined = dec.points_dec[dec.declined]
    exact = principal_reasons(S["ref_exact"]["all applicants"] - declined, MAX_REASONS)
    whole = S["card_reasons"]["all applicants"]
    print(f"The lesson's scorecard reasons use the reference's average rounded to whole points. Against the unrounded average (the")
    print(f"exact Shapley reasons, by the identity above) they differ in the first reason for {_share(exact, whole, 'first'):.1%} of declined")
    print(f"applicants and in the set of {NOTICE_REASONS} reasons for {_share(exact, whole, 'set'):.1%}: that is how far whole points move the reasons.")


_try("identity", _show_identity, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"), heavy=True)

## 10. The challenger's reasons, and what a sampled background does

For the challenger a declined applicant's shortfall on an input is **minus its Shapley value**
(the attributions are in points, so a negative one lowered the applicant's score against the
background). The same `principal_reasons` then picks the reasons. Everything else you wrote
applies unchanged, which is the point of writing the rules for a matrix of shortfalls.

The reference is the same as the scorecard's: every development row ("all applicants"), or the
development rows at or just above the challenger's own cut-off. Because the trees are tables,
the attributions use the **whole** reference population as the background, which a general model
could not afford. Most work with a model that has no such structure has to use a **sample** of the
reference instead, so the second cell measures what that costs: the share of applicants whose
reasons change when `B` evenly spaced development rows stand in for the population, and when one
such sample is compared with another. One sample is one draw (it shifts every applicant's baseline
at once), so the cell takes `N_SAMPLES` equally valid samples of each size and reports the spread:
that spread is the noise floor, not any single figure.

In [ ]:
def booster_reasons_of(model: Challenger, background_bins: np.ndarray, k: int = MAX_REASONS) -> Callable[[np.ndarray], np.ndarray]:
    """The challenger's reasons as a function of raw inputs: minus the Shapley values, then the top k. Given."""
    def reasons_of(X: np.ndarray) -> np.ndarray:
        phi, _, _ = shapley_trees(model, to_bins(X, model.edges), background_bins)
        return principal_reasons(-phi, k)
    return reasons_of


def _show_booster_reasons() -> None:
    if not _ready("challenger's reasons", "challenger", "card_reasons"):
        return
    ch, dec = S["challenger"], S["decision"]
    model = ch["model"]
    idx = np.flatnonzero(ch["declined"])
    apps = X_DEC[idx]
    Xb_dev = to_bins(X_DEV, model.edges)
    band = np.flatnonzero(reference_mask(ch["points_dev"], ch["cut"], MARGINAL_WIDTH))
    backgrounds = {"all applicants": Xb_dev, "at or just above the cut-off": Xb_dev[band]}
    reasons = {name: booster_reasons_of(model, bg)(apps) for name, bg in backgrounds.items()}
    S["boost"] = {"idx": idx, "apps": apps, "reasons": reasons, "backgrounds": backgrounds}
    n = len(idx)
    print(f"challenger: {n:,} declined applicants on the decision rows; the background is the whole reference population:")
    print(f"{N_DEV:,} development rows for 'all applicants', {len(band):,} at or just above its cut-off.\n")
    print(f"share whose FIRST reason is:{'':<5}" + "".join(f"{name:>12}" for name in SHORT) + f"{'no reason':>12}")
    for name, r in reasons.items():
        counts = np.bincount(r[:, 0] + 1, minlength=P + 1)
        print(f"{name:<32}" + "".join(f"{counts[c + 1] / n:>12.1%}" for c in range(P)) + f"{counts[0] / n:>12.1%}")
    moved = {how: _share(reasons["all applicants"], reasons["at or just above the cut-off"], how) for how in ("first", "ordered", "set")}
    S["boost"]["ref_flip"] = moved
    print(f"\nshare of those applicants whose reasons (first {NOTICE_REASONS} places) change when ONLY the reference changes, from 'all applicants'")
    print("to 'at or just above the cut-off':   " + ",  ".join(f"{how} {v:.1%}" for how, v in moved.items()))
    card_first = np.full(N_DEC, -2)
    card_first[np.flatnonzero(dec.declined)] = S["card_reasons"]["all applicants"][:, 0]
    boost_first = np.full(N_DEC, -2)
    boost_first[idx] = reasons["all applicants"][:, 0]
    both = (card_first > -2) & (boost_first > -2)
    print(f"\n{int(both.sum()):,} applicants are declined by both models; the two give the same first reason for "
          f"{np.mean(card_first[both] == boost_first[both]):.1%} of them.")
    print("Each model's reasons describe that model, so they need not agree.")


_try("challenger's reasons", _show_booster_reasons, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"), heavy=True)


def _spread(values: np.ndarray) -> tuple[float, float, float]:
    """(smallest, median, largest) of a set of shares. Given."""
    return float(values.min()), float(np.median(values)), float(values.max())


def _show_background_sampling() -> None:
    if not _ready("sampled backgrounds", "boost"):
        return
    model, apps = S["challenger"]["model"], S["boost"]["apps"]
    Xb_dev = to_bins(X_DEV, model.edges)
    pool = np.arange(N_DEV)
    exact = S["boost"]["reasons"]["all applicants"][:, :NOTICE_REASONS]
    print(f"{len(apps):,} declined applicants, reference 'all applicants'. For each size B, {N_SAMPLES} equally valid samples of B evenly spaced")
    print(f"development rows (the spacing starts at a different row each time) stand in for all {N_DEV:,}. Share of applicants whose reasons")
    print("change, as the smallest, median and largest over those samples:")
    S["sampling"] = {}
    for m in BG_SWEEP:
        step = max(N_DEV // m, 1)
        samples = [booster_reasons_of(model, Xb_dev[systematic_sample(pool, m, (j * step) // N_SAMPLES)], NOTICE_REASONS)(apps)
                   for j in range(N_SAMPLES)]
        S["sampling"][m] = {
            "first, vs exact": np.array([_share(r, exact, "first") for r in samples]),
            "set, vs exact": np.array([_share(r, exact, "set") for r in samples]),
            "first, two samples": np.array([_share(samples[j], samples[(j + 1) % N_SAMPLES], "first") for j in range(N_SAMPLES)]),
            "set, two samples": np.array([_share(samples[j], samples[(j + 1) % N_SAMPLES], "set") for j in range(N_SAMPLES)])}
    for title, keys in (("a sample against the whole population", ("first, vs exact", "set, vs exact")),
                        ("one sample against the next", ("first, two samples", "set, two samples"))):
        print(f"\n{title}:")
        print(f"{'':>8}{'first reason changes':^30}{'reason set changes':^30}")
        print(f"{'B':>8}" + f"{'smallest':>10}{'median':>10}{'largest':>10}" * 2)
        for m in BG_SWEEP:
            print(f"{m:>8}" + "".join(f"{v:>10.1%}" for k in keys for v in _spread(S["sampling"][m][k])))
    lo, med, hi = _spread(S["sampling"][N_BG]["first, vs exact"])
    ref = S["boost"]["ref_flip"]["first"]
    medians = [float(np.median(S["sampling"][m]["first, vs exact"])) for m in BG_SWEEP]
    falls = all(a > b for a, b in zip(medians, medians[1:]))
    print(f"\nThe median first-reason error {'falls at every step' if falls else 'does not fall at every step'} as B grows, from {medians[0]:.1%} at "
          f"B = {BG_SWEEP[0]} to {medians[-1]:.1%} at B = {BG_SWEEP[-1]}.")
    print(f"With {N_BG} rows the choice of sample alone changes the first reason of a median {med:.1%} of applicants against the exact reference "
          f"(smallest {lo:.1%}, largest {hi:.1%}, over {N_SAMPLES} samples).")
    if ref > hi:
        reading = f"above the error of every one of the {N_SAMPLES} samples, {ref / hi:.1f} times the largest"
    elif ref >= lo:
        reading = "inside the range of the samples' errors, so a sample of this size could not tell that reference effect from sampling noise"
    else:
        reading = "below the error of every sample at this size"
    print(f"A change of reference changes it for {ref:.1%} (book 0): {reading}.")
    print("A reference effect measured with a sampled background has to be read against the spread of the sampling error at that size,")
    print(f"not against one draw. (With all {N_DEV:,} rows the background is the population itself.)")


_try("sampled backgrounds", _show_background_sampling, needs=("exercise 3", "exercise 4"), heavy=True)

And the perturbation test on the challenger: the same function you wrote for the scorecard, with
the Shapley reasons behind it. Each call of the reason function recomputes the attributions of all
the declined applicants, so the cell keeps the results it has already computed and compares the
three kinds of change against one set of moves.

In [ ]:
def _memoised(fn: Callable[[np.ndarray], np.ndarray]) -> Callable[[np.ndarray], np.ndarray]:
    """Cache `fn` by the contents of its argument: three comparisons share one set of moved rows. Given."""
    seen: dict = {}

    def cached(X: np.ndarray) -> np.ndarray:
        key = (X.shape, X.tobytes())
        if key not in seen:
            seen[key] = fn(X)
        return seen[key]
    return cached


def _show_booster_perturbation() -> None:
    if not _ready("challenger perturbation", "boost", "card_reasons"):
        return
    model, apps = S["challenger"]["model"], S["boost"]["apps"]
    reasons_of = _memoised(booster_reasons_of(model, S["boost"]["backgrounds"]["all applicants"], NOTICE_REASONS))
    flips = {how: perturbation_flip_rate(reasons_of, apps, CONTINUOUS, SD_DEV, EPS_MAIN, how)[0] for how in ("first", "ordered", "set")}
    bins = to_bins(apps, model.edges)
    changed = np.zeros(len(apps), dtype=bool)
    for j in CONTINUOUS:
        for direction in (-1.0, 1.0):
            moved = apps.copy()
            moved[:, j] += direction * EPS_MAIN * SD_DEV[j]
            changed |= (to_bins(moved, model.edges) != bins).any(axis=1)
    S["boost"]["flip"] = flips | {"changed": float(changed.mean())}
    card = S.get("card_flip", {}).get(EPS_MAIN)
    print(f"a move of {EPS_MAIN:g} of a development standard deviation, up and down, on utilisation, age of account and area index:")
    print(f"{'':<14}{'bin or points of some input changed':>38}{'first reason':>14}{'ordered list':>14}{'set of reasons':>16}")
    if card:
        print(f"{'scorecard':<14}{card['changed']:>38.1%}{card['first']:>14.1%}{card['ordered']:>14.1%}{card['set']:>16.1%}")
    print(f"{'challenger':<14}{changed.mean():>38.1%}{flips['first']:>14.1%}{flips['ordered']:>14.1%}{flips['set']:>16.1%}")
    verdict = "within" if flips["set"] <= STABILITY_TOL else "above"
    print(f"\nThe challenger's reason set changes for {flips['set']:.1%} of these applicants: {verdict} the audit's tolerance of "
          f"{STABILITY_TOL:.0%}, which the lesson chose and the regulation does not set.")


_try("challenger perturbation", _show_booster_perturbation,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6"), heavy=True)

## 11. Exercise 7 — `citation_counts()`: who is cited?

So far the group label has played no part: no model sees it. A test of the *reasons* needs it.
The question is a plain one about the notices: among the group-1 applicants who were declined,
how many were given each reason, and among the group-0 applicants? An applicant counts once for a
code however many of their places it fills, and the places that hold `NO_REASON` count for nothing.
The group sizes come back with the counts because a rate needs its denominator, and a group
with nobody in it is a size of zero, not an error.

<details><summary>💡 Hint 1 — what to think about</summary>

The reasons array has one row per applicant and one column per place on the notice, so "this
applicant cites code c" means "c appears somewhere in this row". Counting the cells equal to `c`
instead would count an applicant twice if a code could fill two places. What should the size of
a group be for an applicant with no reasons at all?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the shapes and values first: reasons is an integer array, group has one entry per row and
holds only 0 and 1, and every reason is -1 or a code below `n_codes`. Then, for each group and
each code, count the rows of that group in which the code appears anywhere. The sizes are the
numbers of rows in each group.

</details>

In [ ]:
def citation_counts(reasons: np.ndarray, group: np.ndarray, n_codes: int) -> tuple[np.ndarray, np.ndarray]:
    """How many applicants of each group are given each code among their reasons.

    Requirements, each graded:
      * counts[g, c] is the number of applicants in group g (0 or 1) with code c ANYWHERE in their row
        of `reasons`: an applicant counts once per code, however many places it fills;
      * NO_REASON (-1) entries count for nothing;
      * sizes[g] is the number of applicants in group g, with or without a reason;
      * a group with nobody in it has a row of zeros and a size of 0, not an error;
      * counts has shape (2, n_codes) and sizes shape (2,), both integer arrays;
      * `ValueError` if `reasons` is not a 2-D integer array, if `group` is not a 1-D array with one
        entry per row holding only 0 and 1, if `n_codes` is below 1, or if a reason is outside
        -1 .. n_codes - 1.

    Example (three applicants; the first two are in group 1, the third in group 0):
        >>> counts, sizes = citation_counts(np.array([[0, 1], [0, -1], [1, -1]]), np.array([1, 1, 0]), 3)
        >>> counts.tolist(), sizes.tolist()
        ([[0, 1, 0], [2, 1, 0]], [1, 2])

    Returns:
        (counts, sizes).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_citation_counts() -> None:
    counts, sizes = citation_counts(np.array([[0, 1], [0, -1], [1, -1]]), np.array([1, 1, 0]), 3)
    assert np.asarray(counts).tolist() == [[0, 1, 0], [2, 1, 0]] and np.asarray(sizes).tolist() == [1, 2], (
        f"got counts {np.asarray(counts).tolist()} and sizes {np.asarray(sizes).tolist()} for the docstring's example, "
        "expected [[0, 1, 0], [2, 1, 0]] and [1, 2]: row = group, column = code; -1 counts for nothing")
    counts, sizes = citation_counts(np.array([[2, 2, 1], [-1, -1, -1]]), np.array([0, 0]), 3)
    assert np.asarray(counts).tolist() == [[0, 1, 1], [0, 0, 0]] and np.asarray(sizes).tolist() == [2, 0], (
        f"got counts {np.asarray(counts).tolist()} and sizes {np.asarray(sizes).tolist()}: an applicant counts ONCE per code "
        "(the first applicant cites code 2 in two places), an applicant with no reason still belongs to the group size, "
        "and the empty group has zeros")
    assert np.issubdtype(np.asarray(counts).dtype, np.integer) and np.asarray(counts).shape == (2, 3), "counts: integers, shape (2, n_codes)"
    for bad in ((np.array([[0.5]]), np.array([0]), 2), (np.array([[0, 1]]), np.array([0, 1]), 2), (np.array([[0, 1]]), np.array([2]), 2),
                (np.array([[0, 3]]), np.array([0]), 3), (np.array([[0, -2]]), np.array([0]), 3), (np.array([[0]]), np.array([0]), 0)):
        try:
            citation_counts(*bad)
        except ValueError:
            continue
        raise AssertionError("float reasons, a group of the wrong length or with a value other than 0 and 1, a reason out of range "
                             "or n_codes below 1 must raise ValueError")
    print("exercise 7 looks right")

In [ ]:
_try("exercise 7", _check_citation_counts)

## 12. Exercise 8 — `proxy_flags()`: a screen, not a verdict

A reason **behaves like a proxy** for the group if three things hold together: it is cited
often enough to say anything (`min_cited` citations in all), group-1 applicants are cited for it
at a rate at least `min_ratio` times group 0's, and the input behind it, on its own, separates
the two groups at least `min_gini` (the Gini of the input used as a score for group membership,
2 AUC - 1, taken as a size: `abs`). All three are needed. A rate difference alone can come from
a legitimate difference in risk; separation alone says nothing about the notices; a handful of
citations says nothing at all.

The comparison of rates is made **on the counts, without dividing**: group 1's citation rate
is at least `min_ratio` times group 0's when
`counts[1, c] * sizes[0] >= min_ratio * counts[0, c] * sizes[1]`, evaluated as written (a code
group 0 is never cited for has nothing on the right, so a cited-only-by-group-1 code passes).
Remember what this is: current Regulation B reads proxies in terms of intent
(section 1 above), and nothing here measures intent. A flag is a reason to look.

<details><summary>💡 Hint 1 — what to think about</summary>

Each code gets its own verdict from its own column of `counts` and its own entry of
`group_gini`. A `nan` Gini (one that could not be computed) can never satisfy a comparison, so it
never flags, and the code must not raise. Which of the three conditions are `>=` and which `<`?
They are all "at least".

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the shapes and the three thresholds (the ratio positive, the Gini not negative, the number
of citations a whole number of at least one). Both groups must have somebody in them or a rate means nothing, so refuse a size
of zero. Then for each code, in whole numbers, check the citation total, then the cross-multiplied
rates, then the absolute Gini against `min_gini`, and flag the code only if all three hold.

</details>

In [ ]:
def proxy_flags(counts: np.ndarray, sizes: np.ndarray, group_gini: np.ndarray, min_ratio: float,
                min_gini: float, min_cited: int) -> np.ndarray:
    """Which reason codes behave like a proxy for group 1.

    `counts` and `sizes` are as `citation_counts` returns them; `group_gini[c]` is how well the input
    behind code c separates the groups. Code c is flagged when ALL of these hold:
      * counts[0, c] + counts[1, c] >= min_cited;
      * counts[1, c] * sizes[0] >= min_ratio * counts[0, c] * sizes[1], evaluated as written on whole
        numbers and the given float (group 1's citation rate is at least min_ratio times group 0's);
      * abs(group_gini[c]) >= min_gini; a nan never satisfies this, and never raises.
    Requirements, each graded: equality passes every comparison; a code only group 1 is cited for passes
    the rate comparison; the result is a boolean array, one entry per code; `ValueError` if `counts` is
    not an integer array of shape (2, m) with no negative entry, `sizes` not two positive whole numbers,
    `group_gini` not of length m, `min_ratio` not positive (a nan is not positive), `min_gini` negative
    or nan, or `min_cited` not a whole number of at least 1.

    Example (code 0 is cited three times as often for group 1 and its input separates the groups; code
    1 is cited equally, so it is not flagged; both groups have 100 applicants):
        >>> proxy_flags(np.array([[10, 40], [30, 40]]), np.array([100, 100]), np.array([0.6, 0.6]), 1.5, 0.25, 30).tolist()
        [True, False]

    Returns:
        a numpy boolean array of length m.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_proxy_flags() -> None:
    counts, sizes = np.array([[10, 40], [30, 40]]), np.array([100, 100])
    got = np.asarray(proxy_flags(counts, sizes, np.array([0.6, 0.6]), 1.5, 0.25, 30)).tolist()
    assert got == [True, False], (f"got {got} for the docstring's example, expected [True, False]: code 0 has "
                                  "30 * 100 >= 1.5 * 10 * 100, code 1 has 40 * 100 < 1.5 * 40 * 100")
    sz = np.array([200, 100])
    only = np.asarray(proxy_flags(np.array([[0], [30]]), sz, np.array([0.5]), 2.0, 0.25, 30)).tolist()
    assert only == [True], f"got {only}: a code that only group 1 is ever cited for has nothing on the right-hand side and passes"
    skew = np.asarray(proxy_flags(np.array([[20], [20]]), sz, np.array([0.5]), 1.5, 0.25, 30)).tolist()
    assert skew == [True], (f"got {skew}: the groups are different sizes, so equal COUNTS are not equal RATES (group 1 "
                            "is cited for 20 of 100 and group 0 for 20 of 200: a ratio of 2)")
    for gini in (0.2, float("nan"), -0.3, -0.25):
        r = np.asarray(proxy_flags(np.array([[10], [30]]), np.array([100, 100]), np.array([gini]), 1.5, 0.25, 30)).tolist()
        want = [bool(abs(gini) >= 0.25)]
        assert r == want, (f"group Gini {gini}: got {r}, expected {want}; the size of the Gini counts (abs), at least min_gini "
                           "passes, and a nan neither passes nor raises")
    few = np.asarray(proxy_flags(np.array([[0], [29]]), np.array([100, 100]), np.array([0.9]), 1.5, 0.25, 30)).tolist()
    assert few == [False], f"got {few}: 29 citations are fewer than min_cited = 30"
    for bad in ((np.array([[1.5], [2.5]]), np.array([1, 1]), np.array([0.5]), 1.5, 0.25, 1),
                (np.array([[1], [2]]), np.array([0, 5]), np.array([0.5]), 1.5, 0.25, 1),
                (np.array([[1], [2]]), np.array([5, 5]), np.array([0.5, 0.5]), 1.5, 0.25, 1),
                (np.array([[1], [2]]), np.array([5, 5]), np.array([0.5]), 0.0, 0.25, 1),
                (np.array([[1], [2]]), np.array([5, 5]), np.array([0.5]), 1.5, -0.1, 1),
                (np.array([[1], [2]]), np.array([5, 5]), np.array([0.5]), 1.5, 0.25, 0),
                (np.array([[1], [2]]), np.array([5, 5]), np.array([0.5]), 1.5, 0.25, 2.5)):
        try:
            proxy_flags(*bad)
        except ValueError:
            continue
        raise AssertionError("float counts, a group of size 0, a Gini vector of the wrong length, a ratio that is not positive, "
                             "a negative Gini threshold or min_cited below 1 or not whole must raise ValueError")
    print("exercise 8 looks right")

In [ ]:
_try("exercise 8", _check_proxy_flags)

In [ ]:
def screen_table(label: str, reasons: np.ndarray, group: np.ndarray, group_gini: np.ndarray, k: int = NOTICE_REASONS) -> np.ndarray:
    """Print the citation table and the flags for one model's reasons, return the flags. Given; uses your exercises 7 and 8."""
    counts, sizes = citation_counts(reasons[:, :k], group, P)
    flags = proxy_flags(counts, sizes, group_gini, MIN_RATIO, MIN_GROUP_GINI, MIN_CITED)
    print(f"{label}: {int(sizes[0]):,} declined applicants in group 0, {int(sizes[1]):,} in group 1; {k} reasons per notice")
    print(f"{'input':<24}{'group Gini':>11}{'cited, g0':>11}{'cited, g1':>11}{'ratio':>8}{'flagged':>9}")
    for j, name in enumerate(FEATURES):
        r0, r1 = counts[0, j] / sizes[0], counts[1, j] / sizes[1]
        ratio = f"{r1 / r0:.2f}" if r0 > 0 else ("inf" if r1 > 0 else "-")
        print(f"{name:<24}{group_gini[j]:>11.3f}{r0:>11.1%}{r1:>11.1%}{ratio:>8}{'  FLAG' if flags[j] else '':>9}")
    return flags


def dilution_table(label: str, reasons: np.ndarray, group: np.ndarray, group_gini: np.ndarray) -> None:
    """Print how the flags change as the notice lists more reasons. Given; uses your exercises 7 and 8."""
    print(f"{label}: inputs flagged as more reasons are listed")
    for k in range(1, MAX_REASONS + 1):
        counts, sizes = citation_counts(reasons[:, :k], group, P)
        flags = proxy_flags(counts, sizes, group_gini, MIN_RATIO, MIN_GROUP_GINI, MIN_CITED)
        ratio = (counts[1] / sizes[1]) / np.maximum(counts[0] / sizes[0], 1e-12)
        names = ", ".join(FEATURES[j] for j in range(P) if flags[j]) or "none"
        print(f"  {k} reason{'s' if k > 1 else ' '}  rate ratio of the area index {ratio[AREA]:.2f}; flagged: {names}")


def _show_card_screen() -> None:
    if not _ready("scorecard screen", "decision", "card_reasons"):
        return
    dec = S["decision"]
    group = G_DEC[dec.declined]
    S["group_gini"] = np.array([abs(auc_gini(X_DEC[:, j], G_DEC)[1]) for j in range(P)])
    print("group Gini: how well each input ALONE separates the two groups, over all decision applicants (not only the declined).")
    S["card_flags"] = screen_table("scorecard, reference 'all applicants'", S["card_reasons"]["all applicants"], group, S["group_gini"])
    print()
    dilution_table("scorecard", S["card_reasons"]["all applicants"], group, S["group_gini"])
    print("\nA flag is a screen on three counts. It says that this reason is cited far more for one group, that its")
    print("input separates the groups, and that it is cited often. It does not say why the model uses the input.")


_try("scorecard screen", _show_card_screen,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 7", "exercise 8"))


def _show_boost_screen() -> None:
    if not _ready("challenger screen", "boost", "group_gini"):
        return
    group = G_DEC[S["boost"]["idx"]]
    S["boost_flags"] = screen_table("challenger, reference 'all applicants'", S["boost"]["reasons"]["all applicants"], group, S["group_gini"])
    print()
    dilution_table("challenger", S["boost"]["reasons"]["all applicants"], group, S["group_gini"])


_try("challenger screen", _show_boost_screen, needs=("exercise 3", "exercise 7", "exercise 8"), heavy=True)

## 13. The audit, and what dropping the flagged input costs

The promise of this module is a stable, non-proxy reason for every decline, **asserted by a test**. The cell after this one is that test, built from your functions: it passes a
configuration only if (1) every declined applicant has at least one reason, (2) the share of
declined applicants whose reason *set* changes under the stated perturbation is within
`STABILITY_TOL` (a float comparison, "at most", so a rate of exactly 5% passes), and (3) no reason is flagged. The first is not a hope: the shortfalls add up to
the reference's total minus the applicant's total, so it holds whenever the declined are below
that total; the audit checks it rather than assuming it. The tolerance and the thresholds of the
screen are this lesson's own choices. The audit reads the screen of exercise 8, which refuses a group
with nobody declined in it (a size of 0); on the books here each group has hundreds of declined
applicants, and a configuration that declined nobody from a group would need a "no one to test" answer first.

It is then run on four configurations: the scorecard and the challenger as they are, and each
**refitted without the area index**. Dropping an input is one remedy and not a free one, so the
table also carries what it costs in Gini and what it does to the gap between the two groups'
decline rates. The lesson does not say which configuration to ship: that is a decision for the
people accountable for the model, with a lawyer, and these are the numbers they would ask for.

In [ ]:
def audit_reasons(reasons: np.ndarray, group: np.ndarray, set_flip_rate: float, group_gini: np.ndarray) -> dict:
    """The test: every decline has a reason, the reason set is stable, no reason is flagged. Given; uses exercises 7 and 8."""
    n_codes = len(group_gini)
    counts, sizes = citation_counts(reasons[:, :NOTICE_REASONS], group, n_codes)
    flags = proxy_flags(counts, sizes, group_gini, MIN_RATIO, MIN_GROUP_GINI, MIN_CITED)
    return {"every decline has a reason": bool((reasons[:, 0] != NO_REASON).all()),
            "reason set stable": bool(set_flip_rate <= STABILITY_TOL),
            "no reason flagged": not bool(flags.any()),
            "flagged": [FEATURES[j] for j in range(n_codes) if flags[j]]}


def _decline_gap(declined: np.ndarray, group: np.ndarray = G_DEC) -> float:
    """Decline rate of group 1 minus group 0 on the decision rows, in percentage points. Given."""
    return 100.0 * float(declined[group == 1].mean() - declined[group == 0].mean())


def _refit_without_area() -> None:
    if not _ready("refits", "card", "decision"):
        return
    keep = [j for j in range(P) if j != AREA]
    X4_dev, X4_dec = X_DEV[:, keep], X_DEC[:, keep]
    out = {}
    card4 = fit_scorecard(X4_dev, Y_DEV, DIRECTIONS[keep])
    dec4 = decide(card4, X4_dev, X4_dec)
    ref4 = whole_reference(dec4.points_dev, np.ones(N_DEV, dtype=bool))
    short4 = ref4 - dec4.points_dec[dec4.declined]
    reasons4 = principal_reasons(short4, MAX_REASONS)
    flip4 = perturbation_flip_rate(card_reasons_of(card4, ref4, NOTICE_REASONS), X4_dec[dec4.declined],
                                   [j for j in CONTINUOUS if j in keep], SD_DEV[keep], EPS_MAIN, "set")[0]
    out["scorecard without the area index"] = {"gini": auc_gini(-dec4.total_dec, Y_DEC)[1], "gap": _decline_gap(dec4.declined),
                                               "reasons": reasons4, "group": G_DEC[dec4.declined], "flip": flip4, "n": int(dec4.declined.sum())}
    model4 = fit_challenger(X4_dev, Y_DEV, BOOK.X[VAL][:, keep], BOOK.y[VAL], DIRECTIONS[keep])
    p_dev, p_dec = points_from_raw(model4, X4_dev), points_from_raw(model4, X4_dec)
    cut4 = cut_from_development(p_dev, DECLINE_SHARE)
    idx = np.flatnonzero(p_dec < cut4)
    bg = to_bins(X4_dev, model4.edges)
    flip = perturbation_flip_rate(booster_reasons_of(model4, bg, NOTICE_REASONS), X4_dec[idx],
                                  [j for j in CONTINUOUS if j in keep], SD_DEV[keep], EPS_MAIN, "set")[0]
    out["challenger without the area index"] = {
        "gini": auc_gini(-p_dec, Y_DEC)[1], "gap": _decline_gap(p_dec < cut4),
        "reasons": booster_reasons_of(model4, bg, MAX_REASONS)(X4_dec[idx]), "group": G_DEC[idx], "flip": flip, "n": int(idx.size)}
    S["refit"] = out
    print("refitted on the four remaining inputs, with the same settings, the same cut-off rule and the same reference ('all applicants').")


_try("refits", _refit_without_area, needs=("exercise 1", "exercise 3", "exercise 4", "exercise 6"), heavy=True)


def _show_audit() -> None:
    if not _ready("audit", "decision", "card_reasons", "challenger", "boost", "group_gini", "refit"):
        return
    dec, ch = S["decision"], S["challenger"]
    gini_full = S["group_gini"]
    rows = {
        "scorecard": {"gini": auc_gini(-dec.total_dec, Y_DEC)[1], "gap": _decline_gap(dec.declined),
                      "reasons": S["card_reasons"]["all applicants"], "group": G_DEC[dec.declined],
                      "flip": S["card_flip"][EPS_MAIN]["set"], "n": int(dec.declined.sum())},
        "challenger": {"gini": auc_gini(-ch["points_dec"], Y_DEC)[1], "gap": _decline_gap(ch["declined"]),
                       "reasons": S["boost"]["reasons"]["all applicants"], "group": G_DEC[S["boost"]["idx"]],
                       "flip": S["boost"]["flip"]["set"], "n": int(ch["declined"].sum())},
    } | S["refit"]
    verdicts = {}
    print(f"{'configuration':<36}{'Gini':>7}{'gap, pts':>9}{'declined':>10}{'set flips':>11}   flagged")
    for name, r in rows.items():
        g_gini = gini_full if "without" not in name else gini_full[[j for j in range(P) if j != AREA]]
        verdicts[name] = audit_reasons(r["reasons"], r["group"], r["flip"], g_gini)
        flagged = ", ".join(verdicts[name]["flagged"]) or "none"
        print(f"{name:<36}{r['gini']:>7.3f}{r['gap']:>+9.1f}{r['n']:>10,}{r['flip']:>11.1%}   {flagged}")
    S["audit"] = verdicts
    print(f"\nthe audit (the reason set may change for at most {STABILITY_TOL:.0%} of declined applicants at a move of {EPS_MAIN:g} sd;")
    print(f"a reason is flagged at a rate ratio of {MIN_RATIO:g}, a group Gini of {MIN_GROUP_GINI:g} and {MIN_CITED} citations):")
    for name, v in verdicts.items():
        checks = {k: val for k, val in v.items() if k != "flagged"}
        what_happened = {"every decline has a reason": "a declined applicant has no reason",
                         "reason set stable": f"the reason set changes for {rows[name]['flip']:.1%} of declined applicants, above the tolerance",
                         "no reason flagged": "a reason is flagged: " + ", ".join(v["flagged"])}
        failed = [what_happened[k] for k, ok in checks.items() if not ok]
        print(f"  {name:<36}{'PASS' if not failed else 'FAIL'}" + (f"  ({'; '.join(failed)})" if failed else ""))
    for name in ("scorecard without the area index", "challenger without the area index"):
        if name in rows:
            ref = rows["scorecard" if name.startswith("scorecard") else "challenger"]
            print(f"\n{name}: Gini {rows[name]['gini'] - ref['gini']:+.3f} against the model with the input, "
                  f"decline-rate gap {rows[name]['gap'] - ref['gap']:+.1f} percentage points.")
    print("\nWhat the table can and cannot say. It can say which configurations pass the three checks as set above. It cannot")
    print("say that a passing one is lawful or that a failing one is not, and it cannot say whether the Gini given up is worth")
    print("the change in the gap: both belong to the people who answer for the model.")


_try("audit", _show_audit, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6", "exercise 7", "exercise 8"),
     heavy=True)

## 14. One book is one draw

Everything above comes from one generated book, and a flip rate, a rate ratio and a Gini are all
estimates from it. The cell below runs the whole pipeline again on `REPLICATES` books from the same
generator (the first is the book you have been using), with your functions doing the work each time:
both models, both references, the perturbation test, the screen, and the models refitted without the
area index. It prints which numbers hold still from book to book and which move. A conclusion that
only one book supports is not one the next book owes you.

In [ ]:
def measure_book(seed: int) -> dict:
    """Everything this lesson measures, for the book with this seed. Given; uses your functions."""
    book = application_book(seed)
    Xd, yd, Xv, yv = book.X[DEV], book.y[DEV], book.X[VAL], book.y[VAL]
    Xt, yt, gt = book.X[DEC], book.y[DEC], book.group[DEC]
    sd = Xd.std(axis=0)
    group_gini = np.array([abs(auc_gini(Xt[:, j], gt)[1]) for j in range(P)])
    full, kept = list(range(P)), [j for j in range(P) if j != AREA]

    def side(kind: str, cols: list) -> dict:
        Xd_c, Xt_c = Xd[:, cols], Xt[:, cols]
        cont = [j for j in CONTINUOUS if j in cols]
        if kind == "scorecard":
            card = fit_scorecard(Xd_c, yd, DIRECTIONS[cols])
            dec = decide(card, Xd_c, Xt_c)
            band = reference_mask(dec.total_dev, dec.cut, MARGINAL_WIDTH)
            ref_all, ref_marg = whole_reference(dec.points_dev, np.ones(N_DEV, dtype=bool)), whole_reference(dec.points_dev, band)
            pts = dec.points_dec[dec.declined]
            reasons = {"all": principal_reasons(ref_all - pts, MAX_REASONS), "marg": principal_reasons(ref_marg - pts, MAX_REASONS)}
            declined, gini = dec.declined, auc_gini(-dec.total_dec, yt)[1]
            flip = perturbation_flip_rate(card_reasons_of(card, ref_all, NOTICE_REASONS), Xt_c[declined], cont, sd[cols], EPS_MAIN, "set")[0]
        else:
            model = fit_challenger(Xd_c, yd, Xv[:, cols], yv, DIRECTIONS[cols])
            p_dev, p_dec = points_from_raw(model, Xd_c), points_from_raw(model, Xt_c)
            cut = cut_from_development(p_dev, DECLINE_SHARE)
            declined, gini = p_dec < cut, auc_gini(-p_dec, yt)[1]
            bins = to_bins(Xd_c, model.edges)
            band = reference_mask(p_dev, cut, MARGINAL_WIDTH)
            reasons = {name: booster_reasons_of(model, bg, MAX_REASONS)(Xt_c[declined]) for name, bg in (("all", bins), ("marg", bins[band]))}
            flip = perturbation_flip_rate(booster_reasons_of(model, bins, NOTICE_REASONS), Xt_c[declined], cont, sd[cols], EPS_MAIN, "set")[0]
        group = gt[declined]
        counts, sizes = citation_counts(reasons["all"][:, :NOTICE_REASONS], group, len(cols))
        flags = proxy_flags(counts, sizes, group_gini[cols], MIN_RATIO, MIN_GROUP_GINI, MIN_CITED)
        ratio = (counts[1, -1] / sizes[1]) / max(counts[0, -1] / sizes[0], 1e-12)
        return {"gini": gini, "gap": _decline_gap(declined, gt), "flip": flip, "ref_flip": _share(reasons["all"], reasons["marg"], "first"),
                "ratio": float(ratio), "flagged": bool(flags.any())}

    out = {}
    for kind in ("scorecard", "challenger"):
        with_area, without = side(kind, full), side(kind, kept)
        out[kind] = {**with_area, "cost": with_area["gini"] - without["gini"], "gap_drop": with_area["gap"] - without["gap"],
                     "flagged_without": without["flagged"]}
    return out


def _sweep_books() -> None:
    runs = [measure_book(_book_seed(k)) for k in range(REPLICATES)]
    S["books"] = runs
    print(f"{REPLICATES} books from the same generator (book 0 is the one above). Stability, with the reasons you wrote:")
    print(f"{'book':>6}{'set flips, scorecard':>22}{'set flips, challenger':>23}{'first reason, ref changed: scorecard':>38}{'challenger':>12}")
    for k, r in enumerate(runs):
        print(f"{k:>6}{r['scorecard']['flip']:>22.1%}{r['challenger']['flip']:>23.1%}{r['scorecard']['ref_flip']:>38.1%}{r['challenger']['ref_flip']:>12.1%}")
    for kind in ("scorecard", "challenger"):
        vals = [r[kind]["ref_flip"] for r in runs]
        rest = vals[1:]
        place = "above all of" if vals[0] > max(rest) else "below all of" if vals[0] < min(rest) else "within the range of"
        print(f"a change of reference changes the {kind}'s first reason for {vals[0]:.1%} of declined applicants in book 0, {place} "
              f"the other books' {min(rest):.1%} to {max(rest):.1%}.")
    print("A reference effect read from one book is one draw: carry the range over books, not the first book's figure.")
    print("\nThe screen, and what dropping the area index does ('Gini lost' and 'gap closed', in Gini and in percentage points):")
    print(f"{'book':>6}{'area ratio, sc':>16}{'ch':>7}{'flagged, sc':>13}{'ch':>7}{'Gini lost, sc':>15}{'ch':>8}{'gap closed, sc':>16}{'ch':>8}")
    for k, r in enumerate(runs):
        sc, ch = r["scorecard"], r["challenger"]
        print(f"{k:>6}{sc['ratio']:>16.2f}{ch['ratio']:>7.2f}{'yes' if sc['flagged'] else 'no':>13}{'yes' if ch['flagged'] else 'no':>7}"
              f"{sc['cost']:>15.3f}{ch['cost']:>8.3f}{sc['gap_drop']:>+16.1f}{ch['gap_drop']:>+8.1f}")
    n_flag = {kind: sum(r[kind]["flagged"] for r in runs) for kind in ("scorecard", "challenger")}
    n_clear = {kind: sum(not r[kind]["flagged_without"] for r in runs) for kind in ("scorecard", "challenger")}
    over = {kind: sum(r[kind]["flip"] > STABILITY_TOL for r in runs) for kind in ("scorecard", "challenger")}
    print(f"\nthe screen flags a reason with the area index in: scorecard {n_flag['scorecard']} of {REPLICATES} books, challenger "
          f"{n_flag['challenger']} of {REPLICATES}; refitted without it, a reason is flagged in {REPLICATES - n_clear['scorecard']} and "
          f"{REPLICATES - n_clear['challenger']} books.")
    print(f"the reason set changes for more than {STABILITY_TOL:.0%} of declined applicants at the stated move in: scorecard {over['scorecard']} "
          f"of {REPLICATES} books, challenger {over['challenger']} of {REPLICATES}.")


_try("replicate books", _sweep_books,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6", "exercise 7", "exercise 8"), heavy=True)

## 15. Common mistakes

- **Leaving ties to the sort.** Whole points tie. A tie broken by whatever the sort
  routine does today gives an applicant a different notice tomorrow, on another machine, or in
  the compiled engine that has to match this one.
- **Rounding with `round`.** Python and numpy send a halfway value to the nearest *even* number,
  so half of the halves go the wrong way against a rule written as "half up", and the C
  implementation that follows will not agree with you.
- **Listing an input that helped.** A shortfall at or below zero is not a reason. Comment
  9(b)(2)-4 forbids leaving out a *principal reason*; it does not ask for a factor that
  counted in the applicant's favour.
- **Reading one background sample as the reference.** Shapley reasons from a sample of the
  reference move when the sample moves, and by a different amount for each equally valid sample.
  Measure that floor at your own size over several samples and read every reference effect
  against its spread, not against one draw.
- **Testing stability by moving one direction only, or by moving a count.** A step function
  changes on one side of its edge. And a count of delinquencies is recorded exactly: the move to
  test is a recording error in a continuous input, not an impossible half a delinquency.
- **Ranking columns when the cause is split.** Two readings of one cause share its points, so
  neither ranks as high as the cause; sum within a code before you rank.
- **Counting cells instead of applicants.** An applicant is cited for a code or not; a code that
  fills two places is still one citation, or the rate is wrong.
- **Treating the flag as a finding, or its absence as clearance.** The screen reports a pattern on
  three counts. It says nothing about why the model uses the input, whether a less different model
  would do as well, or what the regulation's current text makes of either.

Run the cell below to see what a longer notice does to the screen. The numbers are computed on the
spot from the reasons above, with nothing refitted.

In [ ]:
def _show_dilution_note() -> None:
    if not _ready("dilution", "decision", "card_reasons", "group_gini"):
        return
    dec = S["decision"]
    group = G_DEC[dec.declined]
    ratios, cited = [], []
    for k in range(1, MAX_REASONS + 1):
        counts, sizes = citation_counts(S["card_reasons"]["all applicants"][:, :k], group, P)
        ratios.append((counts[1, AREA] / sizes[1]) / (counts[0, AREA] / sizes[0]))
        cited.append(counts.sum() / sizes.sum() / P)
    print("scorecard, as the notice lists 1, 2, ... reasons:")
    print("  rate ratio of the area index:          " + ", ".join(f"{r:.2f}" for r in ratios))
    print("  share of the inputs an applicant cites: " + ", ".join(f"{c:.0%}" for c in cited))
    print("The regulation sets no number of reasons, and the number chosen changes what a screen like this one can see:")
    word = "falls" if ratios[-1] < ratios[0] else "does not fall"
    print(f"In this book the share of the inputs an applicant cites rises from {cited[0]:.0%} to {cited[-1]:.0%} as the notice grows, and the area index's rate ratio "
          f"{word} ({ratios[0]:.2f} with one reason, {ratios[-1]:.2f} with {MAX_REASONS}); a longer notice cites every input for more applicants, and that can thin out a concentration.")


_try("dilution", _show_dilution_note, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 7"))

## 16. Self-check

1. Two inputs have exactly the same shortfall for a declined applicant, and the notice has one place
   left. What makes the notice defensible?
   - (a) either is fine, since they are tied, and both are principal reasons to the same degree
   - (b) whichever the sort routine put first, since that ranking came from the code that scored them
   - (c) a rule fixed in advance, such as the lower input number first, so the notice can be regenerated

2. For an additive scorecard, with the development rows as the background, an input's exact Shapley
   value for one applicant equals:
   - (a) its own points for that applicant minus its mean points over the background
   - (b) its coefficient times the input's standard deviation over the development rows
   - (c) its points divided by the applicant's total points, whatever the background is

3. A background of a few hundred development rows, standing for "all applicants", gives different
   first reasons for some applicants than the whole population would, by a different amount for
   each equally valid sample. Suppose a different reference population changes the first reason of
   more applicants than that spread does. A fair reading is:
   - (a) the choice of reference does not matter, since any sample gives the same attributions
   - (b) the change is real to the extent it exceeds the spread of the sampling error, which falls with size
   - (c) a tree model's attributions cannot be used for reasons, since a tree has no coefficients

4. A reason is flagged by the screen: it is cited far more often for the group under test, its input
   separates the two groups, and it is cited often. The right next step is:
   - (a) conclude the model is unlawful, since a flag on all three counts is itself the proof
   - (b) conclude nothing, since the model never sees the group and the flag must be chance
   - (c) review it: ask what the input adds, what a model without it would cost, and take legal advice

5. Why add up the shortfalls of the inputs that share a reason code before choosing reasons?
   - (a) a cause measured by two columns splits its points, so a two-place notice can name it twice
   - (b) to shorten the notice, so that the reasons fit on one line and meet the rule that it be short
   - (c) because the regulation requires one reason code per input, cited once in the order of the score

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

One last cell, and it is the deliverable. A set of reasons does not hand on as a table: it hands on
with the rule that chose them, the reference it was measured against, the tests it passed and failed,
and the things that were never checked, or someone will later quote it as though it described a real
lender's notices.

In [ ]:
def _reason_note() -> None:
    if not _ready("reason note", "decision", "challenger", "boost", "audit", "card_reasons", "sampling", "books"):
        return
    dec, ch, verdicts = S["decision"], S["challenger"], S["audit"]
    floor_lo, floor_med, floor_hi = _spread(S["sampling"][N_BG]["first, vs exact"])
    books = {kind: [r[kind]["ref_flip"] for r in S["books"]] for kind in ("scorecard", "challenger")}
    print("REASON-CODE NOTE — hand this on with the models")
    print(f"  rule         the principal reasons are the (at most) {NOTICE_REASONS} inputs with the largest strictly positive shortfall; ties go to the")
    print("               lower input number; fewer than that if fewer than that cost the applicant anything")
    print(f"  scorecard    weight-of-evidence classes pooled to each input's documented direction, whole points ({PDO} points double the odds),")
    print(f"               cut-off {dec.cut} (the lowest {DECLINE_SHARE:.0%} of development applicants are declined); shortfall against the reference's average points, in whole points")
    print(f"  challenger   {len(ch['model'].booster.trees)} monotone-constrained trees; exact interventional Shapley attributions on the points scale against "
          "the whole reference population")
    print(f"  references   'all applicants' and 'at or just above the cut-off' (cut-off up to cut-off + {MARGINAL_WIDTH} points), each made of every development row that "
          "qualifies; the reasons")
    print(f"               differ in order for {S['card_ref_flip']['ordered']:.1%} (scorecard) and {S['boost']['ref_flip']['ordered']:.1%} (challenger) of declined applicants "
          f"between them (book 0);")
    print(f"               the first reason changes for {min(books['scorecard']):.1%} to {max(books['scorecard']):.1%} (scorecard) and {min(books['challenger']):.1%} to "
          f"{max(books['challenger']):.1%} (challenger) over {len(S['books'])} books; a {N_BG}-row sample of 'all applicants'")
    print(f"               would change the challenger's first reason for a median {floor_med:.1%} (smallest {floor_lo:.1%}, largest {floor_hi:.1%} over "
          f"{N_SAMPLES} samples)")
    print(f"  stability    a move of {EPS_MAIN:g} development standard deviations, up and down, on the continuous inputs; tolerance {STABILITY_TOL:.0%} of the reason set")
    print(f"  the audit    " + "; ".join(f"{name}: {'PASS' if all(val for k, val in v.items() if k != 'flagged') else 'FAIL'}" for name, v in verdicts.items()))
    print("  known gaps   synthetic data and one book; the group label is assumed available to the test and is never an input; the")
    print("               tolerance and the screen's thresholds are the lesson's own; 'actionable' reasons are not tested; the")
    print("               tree-table attributions are exact for this ensemble and say nothing about a model without that structure; the")
    print("               regulation's text changed in April 2026 and may change again, so re-read it before relying on any sentence here")


_try("reason note", _reason_note, needs=tuple(f"exercise {i}" for i in range(1, 9)), heavy=True)

## What you built, and where it goes next

You now have the step between a score and a notice, written so it can be checked: whole points
whose rounding is a rule, a reference population chosen by the regulation's own two methods, a
choice of reasons with a tie rule, the same choice for a model that is not additive through exact
attributions, and three tests that put numbers on whether a reason survives a hair's breadth,
a change of reference and a look at who it is cited for. The programme's decision engine in C, a
later lesson, is meant to reproduce your integer points and your reason selection exactly, which is
why both were pinned down here to the half point and the tie. Module 5 estimates the group label you
took as given, and the capstone grades whether each reason code survives the counterexamples of
this module.

The habit to carry forward: **a reason is a claim about a model, and a claim that can be
tested should be.** As of 19 October 2026 the regulation asks for reasons that are specific and
principal, accepts any method with results substantially similar to the two it describes, and
reads proxies in terms of intent. A lender that can regenerate the same reasons from the same
applicant, show how far they move for a negligible change and for a change of reference, and say
who they are cited for, has the evidence the next conversation will ask for. Re-read the current
text of 12 CFR part 1002 before you rely on any of this: it has changed this year.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (("exercise 1", _check_class_points, ()),
                                      ("exercise 2", _check_reference_mask, ()),
                                      ("exercise 3", _check_principal_reasons, ()),
                                      ("exercise 4", _check_reasons_differ, ()),
                                      ("exercise 5", _check_code_shortfall, ()),
                                      ("exercise 6", _check_perturbation_flip_rate, ("exercise 4",)),
                                      ("exercise 7", _check_citation_counts, ()),
                                      ("exercise 8", _check_proxy_flags, ())):
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.